<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/las_%EA%B5%AC%ED%98%84_%EC%97%B0%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytorch-pretrained-bert
!pip install wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
import abc
import yaml
import copy
import sys
from functools import partial
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import math
import editdistance as ed
import matplotlib
import argparse
import wandb
import os
from joblib import Parallel, delayed


from tqdm import tqdm
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.distributions.categorical import Categorical
from torch.utils.data import DataLoader
from pytorch_pretrained_bert import BertTokenizer, BertForMaskedLM
from torch.utils.tensorboard import SummaryWriter
matplotlib.use('Agg') #그래프나 파일을 저장할때 사용하는 설정

In [ ]:
os.environ["WANDB_API_KEY"] = "513a1f0c050fa7f60a76b5232e904d8df397082e"
wandb.login()

In [ ]:
name = "LAS_Model_Test"   # ex) WGAN (노션 참고)
id = 'LAS_Original'     # 프로젝트마다 고유 id 부여 (실험마다 다르게 설정해야함, 만약 전 실험을 이어서 진행하고 싶다면 같은 id 기재)
wandb.init(project="LAS_Model_Test", name=name ,id = id, resume = 'allow')

In [3]:
class Timer():
    ''' Timer for recording training time distribution. '''

    def __init__(self):
        self.prev_t = time.time()
        self.clear()

    def set(self):
        self.prev_t = time.time()

    def cnt(self, mode):
        self.time_table[mode] += time.time()-self.prev_t
        self.set()
        if mode == 'bw':
            self.click += 1

    def show(self):
        total_time = sum(self.time_table.values())
        self.time_table['avg'] = total_time/self.click
        self.time_table['rd'] = 100*self.time_table['rd']/total_time
        self.time_table['fw'] = 100*self.time_table['fw']/total_time
        self.time_table['bw'] = 100*self.time_table['bw']/total_time
        msg = '{avg:.3f} sec/step (rd {rd:.1f}% | fw {fw:.1f}% | bw {bw:.1f}%)'.format(
            **self.time_table)
        self.clear()
        return msg

    def clear(self):
        self.time_table = {'rd': 0, 'fw': 0, 'bw': 0}
        self.click = 0

# Reference : https://github.com/espnet/espnet/blob/master/espnet/nets/pytorch_backend/e2e_asr.py#L168


def init_weights(module):
    # Exceptions
    if type(module) == nn.Embedding:
        module.weight.data.normal_(0, 1)
    else:
        for p in module.parameters():
            data = p.data
            if data.dim() == 1:
                # bias
                data.zero_()
            elif data.dim() == 2:
                # linear weight
                n = data.size(1)
                stdv = 1. / math.sqrt(n)
                data.normal_(0, stdv)
            elif data.dim() in [3, 4]:
                # conv weight
                n = data.size(1)
                for k in data.size()[2:]:
                    n *= k
                stdv = 1. / math.sqrt(n)
                data.normal_(0, stdv)
            else:
                raise NotImplementedError


def init_gate(bias):
    n = bias.size(0)
    start, end = n // 4, n // 2
    bias.data[start:end].fill_(1.)
    return bias

# Convert Tensor to Figure on tensorboard


def feat_to_fig(feat):
    # feat TxD tensor
    data = _save_canvas(feat.numpy())
    return torch.FloatTensor(data), "HWC"


def _save_canvas(data, meta=None):
    fig, ax = plt.subplots(figsize=(16, 8))
    if meta is None:
        ax.imshow(data, aspect="auto", origin="lower")
    else:
        ax.bar(meta[0], data[0], tick_label=meta[1], fc=(0, 0, 1, 0.5))
        ax.bar(meta[0], data[1], tick_label=meta[1], fc=(1, 0, 0, 0.5))
    fig.canvas.draw()
    # Note : torch tb add_image takes color as [0,1]
    data = np.array(fig.canvas.renderer._renderer)[:, :, :-1]/255.0
    plt.close(fig)
    return data

# Reference : https://stackoverflow.com/questions/579310/formatting-long-numbers-as-strings-in-python


def human_format(num):
    magnitude = 0
    while num >= 1000:
        magnitude += 1
        num /= 1000.0
    # add more suffixes if you need them
    return '{:3.1f}{}'.format(num, [' ', 'K', 'M', 'G', 'T', 'P'][magnitude])


def cal_er(tokenizer, pred, truth, mode='wer', ctc=False):
    # Calculate error rate of a batch
    if pred is None:
        return np.nan
    elif len(pred.shape) >= 3:
        pred = pred.argmax(dim=-1)
    er = []
    for p, t in zip(pred, truth):
        p = tokenizer.decode(p.tolist(), ignore_repeat=ctc)
        t = tokenizer.decode(t.tolist())
        if mode == 'wer':
            p = p.split(' ')
            t = t.split(' ')
        er.append(float(ed.eval(p, t))/len(t))
    return sum(er)/len(er)


def load_embedding(text_encoder, embedding_filepath):
    with open(embedding_filepath, "r") as f:
        vocab_size, embedding_size = [int(x)
                                      for x in f.readline().strip().split()]
        embeddings = np.zeros((text_encoder.vocab_size, embedding_size))

        unk_count = 0

        for line in f:
            vocab, emb = line.strip().split(" ", 1)
            # fasttext's <eos> is </s>
            if vocab == "</s>":
                vocab = "<eos>"

            if text_encoder.token_type == "subword":
                idx = text_encoder.spm.piece_to_id(vocab)
            else:
                # get rid of <eos>
                idx = text_encoder.encode(vocab)[0]

            if idx == text_encoder.unk_idx:
                unk_count += 1
                embeddings[idx] += np.asarray([float(x)
                                               for x in emb.split(" ")])
            else:
                # Suppose there is only one (w, v) pair in embedding file
                embeddings[idx] = np.asarray(
                    [float(x) for x in emb.split(" ")])

        # Average <unk> vector
        if unk_count != 0:
            embeddings[text_encoder.unk_idx] /= unk_count

        return embeddings

In [4]:
BERT_FIRST_IDX = 997  # Replacing the 2 tokens right before english starts as <eos> & <unk>
BERT_LAST_IDX = 29635  # Drop rest of tokens

class _BaseTextEncoder(abc.ABC): #텍스트에 대한 인코딩
    def encode(self, s):
      raise NotImplementedError

    @abc.abstractmethod
    def decode(self, ids, ignore_repeat=False):
      raise NotImplementedError

    @abc.abstractproperty
    def vocab_size(self):
      raise NotImplementedError

    @abc.abstractproperty
    def token_type(self):
      raise NotImplementedError

    @abc.abstractclassmethod
    def load_from_file(cls, vocab_file):
      raise NotImplementedError

    @property
    def pad_idx(self):
      return 0

    @property
    def eos_idx(self):
      return 1

    @property
    def unk_idx(self):
      return 2

    def __repr__(self):
      return "<{} vocab_size={}>".format(type(self).__name__, self.vocab_size)




In [5]:
class CharacterTextEncoder(_BaseTextEncoder):
    def __init__(self, vocab_list): #인코딩
        self._vocab_list = ["<pad>", "<eos>", "<unk>"] + vocab_list
        self.vocab2idx = {v: idx for idx, v in enumerate(self._vocab_list)}

    def encode(self, s):
        s = s.strip("\r\n")
        return [self.vocab_to_idx(c) for c in s] + [self.eos_idx]

    def decode(self, idxs, ignore_repeat=False):
        vocabs = []
        for t, idx in enumerate(idxs):
            v = self.idx_to_vocab(idx) #id 시퀀스에서 문자열로 변경
            if idx == self.pad_idx or (ignore_repeat and t > 0 and idx == idxs[t - 1]):
                continue
            elif idx == self.eos_idx:
                break
            else:
                vocabs.append(v)
        return "".join(vocabs)

    @classmethod
    def load_from_file(cls, vocab_file): #vocab 불러오기
        with open(vocab_file, "r", encoding="utf-8") as f:
            vocab_list = [line.strip("\r\n") for line in f]
        return cls(vocab_list)

    @property
    def vocab_size(self):
        return len(self._vocab_list)

    @property
    def token_type(self):
        return 'character'

    def vocab_to_idx(self, vocab): #문자 -> id
        return self.vocab2idx.get(vocab, self.unk_idx)

    def idx_to_vocab(self, idx):# id -> 문자
        return self._vocab_list[idx]


In [6]:
class SubwordTextEncoder(_BaseTextEncoder): #서브 워드 단위 인코딩, 디코딩
    def __init__(self, spm):#단어보다 작은 단어 단위
        if spm.pad_id() != 0 or spm.eos_id() != 1 or spm.unk_id() != 2:
            raise ValueError(
                "Please train sentencepiece model with following argument:\n"
                "--pad_id=0 --eos_id=1 --unk_id=2 --bos_id=-1 --model_type=bpe --eos_piece=<eos>"
            )
        self.spm = spm

    def encode(self, s):
        return self.spm.encode_as_ids(s)

    def decode(self, idxs, ignore_repeat=False):
        crop_idx = []
        for t, idx in enumerate(idxs):
            if idx == self.eos_idx:
                break
            elif idx == self.pad_idx or (ignore_repeat and t > 0 and idx == idxs[t - 1]):
                continue
            else:
                crop_idx.append(idx)
        return self.spm.decode_ids(crop_idx)

    @classmethod
    def load_from_file(cls, filepath):
        import sentencepiece as splib
        spm = splib.SentencePieceProcessor()
        spm.load(filepath)
        spm.set_encode_extra_options(":eos")
        return cls(spm)

    @property
    def vocab_size(self):
        return len(self.spm)

    @property
    def token_type(self):
        return 'subword'


In [7]:
class WordTextEncoder(CharacterTextEncoder):#단어 단위 인코더
  def encode(self,s):
    s=s.strip("\r\n")
    words=s.split(" ")
    return [self.vocab_to_idx(v) for v in words] + [self.eos_idx]

  def decode(self,idxs,ignore_repeat=False):
    vocabs=[]
    for t, idx in enumerate(idxs):
      v=self.idx_to_vocab(idx)
      if idx==self.eos_idx:
        break
      elif idx==self.pad_idx or (ignore_repeat and t>0 and idx==idxs[t-1]):
        continue
      else:
        vocabs.append(v)
    return " ".join(vocabs)

  def token_type(self):
    return 'word'

In [8]:
class BertTextEncoder(_BaseTextEncoder):
  def __init__(self,tokenizer):
    self._tokenizer=tokenizer
    self._tokenizer.pad_token="<pad>"
    self._tokenizer.unk_token="<unk>"
    self._tokenizer.eos_token="<eos>"

  def encode(self,s):
    reduced_idx=[]
    for idx in self._tokenizer.encode(s):
      try:
        r_idx=idx-BERT_FIRST_IDX
        assert r_idx>0
        reduced_idx.append(r_idx)
      except:
        reduced_idx.append(self.unk_idx)
    reduced_idx.append(self.eos_idx)
    return reduced_idx

  def decode(self,idxs,ignore_repeat=False):
    crop_idx=[]
    for t,idx in enumerate(idxs):
      if idx==self.eos_idx:
        break
      elif idx==self.pad_idx or (ignore_repeat and t>0 and idx==idxs[t-1]):
        continue
      else:
        crop_idx.append(idx+BERT_FIRST_IDX)
    return self._tokenizer.decode(crop_idx)

  def vocab_size(self):
    return BERT_LAST_IDX-BERT_FIRST_IDX+1

  def token_type(self):
    return 'bert'

  def load_from_file(cls, vocab_file):
        from pytorch_transformers import BertTokenizer
        return cls(BertTokenizer.from_pretrained(vocab_file))

  @property
  def pad_idx(self):
      return 0

  @property
  def eos_idx(self):
      return 1

  @property
  def unk_idx(self):
      return 2

def load_text_encoder(mode, vocab_file):
    if mode == "character":
        return CharacterTextEncoder.load_from_file(vocab_file)
    elif mode == "subword":
        return SubwordTextEncoder.load_from_file(vocab_file)
    elif mode == "word":
        return WordTextEncoder.load_from_file(vocab_file)
    elif mode.startswith("bert-"):
        return BertTextEncoder.load_from_file(mode)
    else:
        raise NotImplementedError("`{}` is not yet supported.".format(mode))

In [9]:
class CMVN(torch.jit.ScriptModule):
  def __init__(self,mode="global",dim=2,eps=1e-10):
    super(CMVN,self).__init__()
    if mode!="global":
      raise NotImplementedError(
                "Only support global mean variance normalization.")

    self.mode=mode
    self.dim=dim
    self.eps=eps

  def forward(self,x):
    if self.mode=="global":
      return (x-x.mean(self.dim,keepdim=True))/(self.eps+x.std(self.dim,keepdim=True))

  def extra_repr(self):
    return "mode={},dim={},eps={}".format(self.mode,self.dim,self.eps)


In [10]:
class Delta(torch.jit.ScriptModule): #델타 특성
  __constants__=["order","window_size","padding"]

  def __init__(self,order=1,window_size=2):
    super(Delta,self).__init__()

    self.order=order
    self.window_size=window_size

    filters=self._create_filters(order,window_size)
    self.register_buffer("filters",filters)
    self.padding=(0,(filters.shape[-1]-1)//2)

  def forward(self,x):
    x=x.squeeze(0)
    return F.conv2d(x,weight=self.filters,padding=self.padding[0])

  def _create_filters(self,order,window_size):
    scales=[[1.0]]
    for i in range(1,order+1):
      prev_offset=(len(scales[i-1])-1)//2
      curr_offset=prev_offset+window_size

      curr=[0]*(len(scales[i-1])+2*window_size)
      normalizer=0.0
      for j in range(-window_size,window_size+1):
        normalizer+=j*j
        for k in range(-prev_offset,prev_offset+1):
          curr[j+k+curr_offset] += (j * scales[i-1][k+prev_offset])
      curr=[x/normalizer for x in curr]
      scales.append(curr)

    max_len=len(scales[-1])
    for i, scales in enumerate(scales[:-1]):
      padding=(max_len-len(scale))//2
      scales[i]=[0]*padding + scale+[0]*padding

    return torch.tensor(scales.unsqueeze(1).unsqueeze(1))

  def extra_repr(self):
    return "order={}, window_size={}".format(self.order, self.window_size)



In [11]:
class Postprocess(torch.jit.ScriptModule):
  def forward(self,x):
    x=x.permute(2,0,1)
    return x.reshape(x.size(0),-1).detach()

In [12]:
# 공부하기

# TODO(Windqaq): make this scriptable
class ExtractAudioFeature(nn.Module):
    def __init__(self, mode="fbank", num_mel_bins=40, **kwargs):
        super(ExtractAudioFeature, self).__init__()
        self.mode = mode
        self.extract_fn = torchaudio.compliance.kaldi.fbank if mode == "fbank" else torchaudio.compliance.kaldi.mfcc
        self.num_mel_bins = num_mel_bins
        self.kwargs = kwargs

    def forward(self, filepath):
        waveform, sample_rate = torchaudio.load(filepath)

        y = self.extract_fn(waveform,
                            num_mel_bins=self.num_mel_bins,
                            channel=-1,
                            sample_frequency=sample_rate,
                            **self.kwargs)
        return y.transpose(0, 1).unsqueeze(0).detach()

    def extra_repr(self):
        return "mode={}, num_mel_bins={}".format(self.mode, self.num_mel_bins)


def create_transform(audio_config):
    feat_type = audio_config.pop("feat_type")
    feat_dim = audio_config.pop("feat_dim")

    delta_order = audio_config.pop("delta_order", 0)
    delta_window_size = audio_config.pop("delta_window_size", 2)
    apply_cmvn = audio_config.pop("apply_cmvn")

    transforms = [ExtractAudioFeature(feat_type, feat_dim, **audio_config)]

    if delta_order >= 1:
        transforms.append(Delta(delta_order, delta_window_size))

    if apply_cmvn:
        transforms.append(CMVN())

    transforms.append(Postprocess())

    return nn.Sequential(*transforms), feat_dim * (delta_order + 1)

In [13]:
# Batch size will be halfed if the longest wavefile surpasses threshold
HALF_BATCHSIZE_AUDIO_LEN = 800
# Note: Bucketing may cause random sampling to be biased (less sampled for those length > HALF_BATCHSIZE_AUDIO_LEN )
HALF_BATCHSIZE_TEXT_LEN = 150

In [14]:
def collect_audio_batch(batch,audio_transform,mode):
  if type(batch[0]) is not tuple:
    batch=batch[0]

  first_len=audio_transform(str(batch[0][0])).shape[0]
  if first_len>HALF_BATCHSIZE_AUDIO_LEN and mode=='train':
    batch_size=batch[:len(batch)//2]

  file,audio_feat,audio_len,text=[],[],[],[]

  with torch.no_grad():
    for b in batch:
      file.append(str(b[0]).split('/')[-1].split('.')[0])
      feat=audio_transform(str(b[0]))
      audio_feat.append(feat)
      text.append(torch.LongTensor(b[1]))
  audio_len, file, audio_feat, text = zip(*[(feat_len, f_name, feat, txt)
                                              for feat_len, f_name, feat, txt in sorted(zip(audio_len, file, audio_feat, text), reverse=True, key=lambda x:x[0])])

  audio_feat=pad_sequence(audio_feat,batch_first=True)
  text=pad_sequence(text,batch_first=True)
  audio_len=torch.LongTensor(audio_len)

  return file, audio_feat,audio_len,text

def collect_text_batch(batch,mode):
  if type(batch[0][0]) is list:
    batch=batch[0]

  if len(batch[0])>HALF_BATCHSIZE_TEXT_LEN and mode=='train':
    batch=batch[:len(batch)//2]

  text=[torch.LongTensor(b) for b in batch]
  text=pad_sequence(text,batch_first=True)

  return text

def create_dataset(tokenizer,ascending,name,path,bucketing, batch_size,train_split=None,dev_split=None, test_split=None):

  if name.lower()=="librispeech":
    from corpus.librispeech import LibriDataset as Dataset
  else:
    raise NotImplementedError

  if train_split is None:
    mode='train'
    tr_loader_bs=1 if bucketing and (not ascending) else batch_size #비슷한 길이끼리 묶어서 메모리 아낌
    bucket_size=batch_size if bucketing and (not ascending) else 1
    dv_set=Dataset(path,dev_split, tokenizer,1)
    tr_set=Dataset(path,train_split,tokenizer, bucket_size,ascending=ascending)

    msg_list = _data_msg(name, path, train_split.__str__(), len(tr_set),
                             dev_split.__str__(), len(dv_set), batch_size, bucketing)

    return tr_set,dv_set,tr_loader_bs, batch_size,mode,msg_list

  else:
    mode="test"
    dv_set=Dataset(path,dev_split,tokenizer,1)
    tt_set=Dataset(path,test_split,tokenizer,1)
    msg_list = _data_msg(name, path, dev_split.__str__(), len(dv_set),
                             test_split.__str__(), len(tt_set), batch_size, False)
    msg_list = [m.replace('Dev', 'Test').replace(
            'Train', 'Dev') for m in msg_list]
    return dv_set,tt_set,batch_size,mode,msg_list


def create_textset(tokenizer,train_split,dev_split,name,path,bucketing,batch_size):
  msg_list=[]

  if name.lower()=="librispeech":
    from corpus.librispeech import LibriDataset as Dataset
  else:
    raise NotImplementedError

  bucket_size=batch_size if bucketing else 1
  tr_loader_bs=1 if bucketing else batch_size
  dv_set=Dataset(path,dev_split,tokenizer,1)
  tr_set=Dataset(path,train_split,tokenizer,bucket_size)

  msg_list = _data_msg(name, path, train_split.__str__(), len(tr_set),
                         dev_split.__str__(), len(dv_set), batch_size, bucketing)

  return tr_set, dv_set,tr_loader_bs, batch_size, msg_list


def load_dataset(n_jobs,use_gpu,pin_memory,ascending,corpus,audio,text):

  audio_transform, feat_dim=create_transform(audio.copy())

  tokenizer=load_text_encoder(**text)
  tr_set,dv_set,tr_loader_bs,dv_loader_bs,mode,data_mst=create_dataset(tokenizer,ascending,**corpus)

  collect_tr=partial(collect_audio_batch,audio_transform=audio_transform,mode=mode)
  collect_dv=partial(collect_audio_batch,audio_transform=audio_transform,mode='test')
  shuffle=(mode=='train' and not ascending)

  drop_last=shuffle

  tr_set=DataLoader(tr_set, batch_size=tr_loader_bs,shuffle=shuffle, drop_last=drop_last,collate_fn=collect_tr,
                    num_workers=n_jobs,pin_memory=use_gpu)
  dv_set=DataLoader(dv_set, batch_size=dv_loader_bs, shuffle=False,drop_last=False,collate_fn=collect_dv,
                    num_workers=n_jobs,pin_memory=pin_memory)
  data_msg.append('I/O spec.  | Audio feature = {}\t| feature dim = {}\t| Token type = {}\t| Vocab size = {}'
                    .format(audio['feat_type'], feat_dim, tokenizer.token_type, tokenizer.vocab_size))

  return tr_set,dv_set,feat_dim,tokenizer.vocab_size,tokenizer,data_msg

def load_textset(n_jobs,use_gpu,pin_memory,corpus,text):
  tokenizer=load_text_encoder(**text)

  tr_set,dv_set,tr_loader_bs,dv_loader_bs,data_msg=create_textset(tokenizer,**corpus)
  collect_tr=partial(collect_text_batch,mode='train')
  collect_dv=partial(collect_text_batch,mode='dev')
  tr_set=DataLoader(tr_set,batch_size=tr_loader_bs,shuffle=True,drop_last=True,collate_fn=collect_tr,
                    num_workers=0,pin_memory=use_gpu)
  dv_set=DataLoader(dv_set,batch_size=dv_loader_bs,shuffle=False,drop_last=False, collate_fn=collect_dv,
                    num_workers=0,pin_memory=pin_memory)
  data_msg.append('I/O spec.  | Token type = {}\t| Vocab size = {}'
                    .format(tokenizer.token_type, tokenizer.vocab_size))

  return tr_set,dv_set,tokenizer.vocab_size,tokenizer,data_msg

def _data_msg(name, path, train_split, tr_set, dev_split, dv_set, batch_size, bucketing):
  ''' List msg for verbose function '''
  msg_list = []
  msg_list.append('Data spec. | Corpus = {} (from {})'.format(name, path))
  msg_list.append('           | Train sets = {}\t| Number of utts = {}'.format(
      train_split, tr_set))
  msg_list.append(
      '           | Dev sets = {}\t| Number of utts = {}'.format(dev_split, dv_set))
  msg_list.append('           | Batch size = {}\t\t| Bucketing = {}'.format(
      batch_size, bucketing))
  return msg_list

In [15]:
class VGGExtractor(nn.Module): #특징 추출기(이미지 기반)

  def __init__(self, input_dim):  # ← 오타 수정 필요 (nn.Module → input_dim)
    super(VGGExtractor, self).__init__()
    self.dim=64
    self.hide_dim=128
    in_channel, freq_dim, out_dim=self.check_dim(input_dim)
    self.in_channel=in_channel
    self.freq_dim=freq_dim
    self.out_dim=out_dim

    self.extractor=nn.Sequential( # b,1,128,80
        nn.Conv2d(in_channel,self.init_dim,3,stride=3,padding=1),# kernel_size=3
        nn.ReLU(),
        nn.Conv2d(self.init_dim,self.init_dim,3,stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,stride=2), #절반으로 줄어듦?


        nn.Conv2d(self.init_dim,self.hide_dim,3,stride=1,padding=1),# kernel_size=3
        nn.ReLU(),
        nn.Conv2d(self.hide_dim,self.hide_dim,3,stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,stride=2) # b,t//4,output_dim
    )

    def check_dim(self,input_dim):
      if input_dim%13==0: #MFCC
        return int(input_dim/13),13,(13//4)*self.hide_dim #TIME, 주파수 단위 크기, 특징 벡터 차원

      elif input_dim%40==0: #LOG MEL
        return int(input_dim/40),40,(40//4)*self.hide_dim

      else:
        raise ValueError('Invalid input dimension')

    def view_input(self,feature,feat_len): #b,t,d -> b,c,t,f
      feat_len=feat_len//4
      if feature.shape[1]%4!=0:
        feature=feature[:,:-(feature.shape[1]%4),:].contiguous()
      bs,ts,ds=feature.shape
      feature=feature.view(bs,ts,self.in_channel,self.freq_dim)
      feature=feature.transpose(1,2)
      return feature, feat_len


    def forward(self,feature, feat_len):
      feature, feat_len=self.view_input(feature,feat_len) #b,t,d -> b,c,t,f
      feature=feature.extractor(feature) #다운샘플링, 고차원 feqture 추출
      feature=feature.transpose(1,2) #b,c,t,f -> b,t,c,f
      feature=feature.contiguous().view(feature.shape[0],feature.shape[1],self.out_dim)
      return feature, feat_len

In [16]:
class CNNExtractor(nn.Module):
  def __init__(self,input_dim,out_dim):
    super(CNNExtractor,self).__init__()
    self.out_dim=out_dim
    self.extractor=nn.Sequential(
        nn.Conv1d(input_dim,out_dim,4,stride=2,padding=1),
        nn.Conv1d(out_dim,out_dim,4,stride=2,padding=1),
    )

  def forward(self,feature,feat_len):
    feat_len=feat_len//4
    feature=feature.transpose(1,2)
    feature=self.extractor(feature)
    feature=feature.transpose(1,2)
    return feature, feat_len

In [17]:
class RNNLayer(nn.Module):
  def __init__(self,input_dim,module,dim,bidirection,dropout,layer_norm,sample_rate,sample_style,proj):
    super(RNNLayer,self).__init__()
    rnn_out_dim=2*dim if bidirection else dim
    self.out_dim=sample_rate * rnn_out_dim if sample_rate>1 and sample_style=='concat' else rnn_out_dim
    self.dropout=dropout
    self.layer_norm=layer_norm
    self.sample_rate=sample_rate
    self.sample_style=sample_style
    self.proj=proj

    if self.sample_style not in ['drop','concat']:
      raise ValueError('Unsupported Sample Style: '+self.sample_style)

    self.layer=getattr(nn.module.upper())(
        input_dim,dim,bidirectional=bidirection,num_layer=1,batch_first=True
    )

    if self.layer_norm:
      self.ln=nn.LayerNorm(rnn_out_dim)
    if self.dropout>0:
      self.dp=nn.Dropout(p=dropout)
    if self.proj:
      self.pj=nn.Linear(rnn_out_dim,rnn_out_dim)

  def forward(self,input_x,x_len):
    if not self.training:
      self.layer.flatten_parameters()
    output,_=self.layer(input_x)

    if self.layer_norm:
      output=self.ln(output)

    if self.dropout>0:
      output=self.dp(output)

    if self.sample_rate>1:
      batch_size,timestep,feature_dim=output.shape
      x_len=x_len//self.sample_rate

      if self.sample_style=='drop':
        output=output[:,::self.sample_rate,:].contiguous()
      else:
        if timestep%self.sample_rate!=0:
          output=output[:,:-(timestep%self.sample_rate),:]
        output=output.contiguous().view(batch_size,int(timestep/self.sample_rate),feature_dim*self.sample_rate)

    if self.proj:
      output=torch.tanh(self.pj(output))

    return output, x_len

In [18]:
class BaseAttention(nn.Module):

  def __init__(self,temprature,num_head):
    super().__init__()
    self.temprature=temprature
    self.num_head=num_head
    self.softmax=nn.Softmax(dim=-1)
    self.reset_mem()

  def reset_mem(self):#마스크 초기화
    self.mask=None
    self.k_len=None

  def set_mem(self,prev_att):
    pass

  def compute_mask(self,k,k_len): #마스크 생성 함수
    self.k_len=k_len #b,t,d 패딩을 제외한 길이 확인
    bs,ts,_=k.shape # b,t
    self.mask=np.zeros((bs,self.num_head,ts)) #b, num_head, t
    for idx,sl in enumerate(k_len):# b,t,d
      self.mask[idx, : , sl:]=1 #마스크 범위 1로 변경
    self.maks=torch.from_numpy(self.mask).to(k_len.device,dtype=torch.bool).view(-1,ts)

  def _attend(self,energy, value): #마스크 씌우기
    attn=energy/self.temperature
    attn=attn.masked_fill(self.mask, -np.inf) #마스크 위치에 -inf 지정
    attn=self.softmax(attn)
    output=torch.bmm(attn.unsqueeze(1),value).squeeze(1)
    return output,attn

In [19]:
class ScaleDotAttention(BaseAttention): # scaledot 연산

  def __init__(self,temperature,num_head):
    super().__init__(temperature,num_head)

  def forward(self,q,k,v): #쿼리, 키, 밸류 텐서
    ts=k.shape[1]
    energy=torch.bmm(q.unsqueeze(1),k.transpose(1,2)).squeeze(1)#배치 단위 행렬곱
    output,attn=self._attend(energy,v)
    attn=attn.view(-1,self.num_head,ts) #attn의 모양을 [batch, num_heads, time_steps]로 변경

    return output,attn

In [20]:
class ASR(nn.Module): #asr 모델
  def __init__(self,input_size,vocab_size,init_adadelta,ctc_weight, encoder,attention, decoder, emb_drop=0.0):
    super(ASR,self).__init__()

    assert 0<=ctc_weight<=1
    self.vocab_size=vocab_size
    self.ctc_weight=ctc_weight
    self.enable_ctc=ctc_weight>0
    self.enable_att=ctc_weight!=1
    self.lm=None

    self.encoder=Encoder(input_size,**encoder) #[b,t',d]
    if self.enable_ctc:
      self.ctc_layer=nn.Linear(self.encoder.out_dim, vocab_size) #[b,t',vocab_size]

    if self.enable_att: #att 분기
      self.dec_dim=decoder['dim'] #디코더의 hidden dimmension 가져옴
      self.pre_embed=nn.Embedding(vocab_size,self.dec_dim) #임베딩 테이블, [b,]-> [b,dec_dim]
      self.embed_drop=nn.Dropout(emb_drop)
      #context=[b,e]
      #embed=[b,d]
      #concat=[b,e+d]
      self.decoder=Decoder(self.encoder.out.dim+self.dec_dim,vocab_size,**decoder)#[b,e+d]
      query_dim=self.dec_dim*self.decoder.layer
      self.attention=Attention(self.encoder.out_dim,query_dim,**attention)
      #key, value = [b,t,endocer.out_dim]
      #query = [b,query_dim]

    if init_adadelta: #adadelta 옵티마이저 사용하는 경우
      self.apply(init_weights) #가중치 적용
      if self.enable_att:
        for l in range(self.decoder.layer):
          bias=getattr(self.decoder.layers,'bias_ih_l{}'.format(l))
          bias=init_gate(bias)

    def set_state(self,prev_state, prev_attn):
      self.decoder.set_state(prev_state) #이전 상태와 메모리 상태 가져옴
      self.attention.set_mem(prev_attn)

    def create_msg(self):
      # Messages for user
      msg = []
      msg.append('Model spec.| Encoder\'s downsampling rate of time axis is {}.'.format(
          self.encoder.sample_rate))
      if self.encoder.vgg:
          msg.append(
              '           | VGG Extractor w/ time downsampling rate = 4 in encoder enabled.')
      if self.encoder.cnn:
          msg.append(
              '           | CNN Extractor w/ time downsampling rate = 4 in encoder enabled.')
      if self.enable_ctc:
          msg.append('           | CTC training on encoder enabled ( lambda = {}).'.format(
              self.ctc_weight))
      if self.enable_att:
          msg.append('           | {} attention decoder enabled ( lambda = {}).'.format(
              self.attention.mode, 1-self.ctc_weight))
      return msg


    #encoder ->  ctc layer + log softmax -> decoder + attention -> 출력
    def forward(self,audio_feature,feature_len,decode_step,tf_rate=0.0,teacher=None,emb_decoder=None, get_dec_state=False):
      bs=audio_feature.shape[0] #배치 사이즈
      ctc_output, att_output,att_seq=None,None,None
      dec_state=[] if get_dec_state else None

      #
      encode_feature, encode_len=self.encoder(audio_feature, feature_len) #[b,t',d]

      if self.enable_ctc:
        ctc_output=F.log_softmax(self.ctc_layer(encode_feature),dim=-1) #[b,t',vocab_size]

      if self.enable_att:
        self.decoder.init_state(bs)
        self.attention.reset_mem()
        last_char = self.pre_embed(torch.zeros((bs), dtype=torch.long, device=encode_feature.device))
        att_seq, output_seq=[],[]

        if teacher is not None: #정답 있으면 임베딩 처리
          teacher=self.embed_drop(self.pre_embed(teacher))


        for t in range(decode_step): #디코딩 루프
          self.decode.init_state(bs)
          self.attention.reset_mem()
          #query 생성 후, attention 수행
          attn,context=self.attention(self.decoder.get_query(),encode_feature,encode_len)

          #임베딩 + context 한 후, decoder 입력
          decoder_input=torch.cat([last_char,context],dim=-1)
          # decoder_input =[b,e+d]
          # cur_char = [b,vocab_size]
          # d_stat= [b,dec_dim*num_layers]
          cur_char, d_stat=self.decoder(decoder_input) #디코더를 통해 다음

          # teacher forcing
          if (teacher is not None):#학습하고 있는지 체크
            if (tf_rate==1) or (torch.rand(1).item()<=tf_rate): #학습중, 정답값 사용확률 지정
              last_char=teacher[:,t,:]
            else: #학습중 아님, 추론중
              with torch.no_grad():
                if(emb_decoder is not None) and emb_decoder.apply_fuse:
                  _, cur_prob=emb_decoder(d_state,cur_char, return_loss=False)

                else:
                  cur_prob=cur_char,softmax(dim=-1)
                sampled=Categorical(cur_prob).sample()
              last_char=self.embed_drop(self.pre_embed(sampled_char))
          else:
            if (emb_decoder is not None) and emb_decoder.apply_fuse:
              _, cur_prob=emb_decoder(d_state, cur_char,return_loss=False)
            last_char=self.pre_embed(torch.argmax(cur_char,dim=-1)) #예측확률 중 가장 높은 토큰 선택 후 임베딩

          output_seq.append(cur_char)
          att_seq.append(attn)
          if get_dec_state: #디코더 상태 저장
            dec_state.append(d_state)

        att_output=torch.stack(output_seq,dim=1) #[b,t,vocab_size]
        att_seq=torch.stack(att_seq,dim=2) #[b,num_head,t]
        if get_dec_state:
          dec_state=torch.stack(dec_state,dim=1) #[b,t,dec_dim]
      return ctc_output,encode_len, att_output,att_seq,dec_state


In [21]:
class Decoder(nn.Module):

  def __init__(self,input_dim,vocab_size,module,dim,layer,dropout):
    super(Decoder,self).__init__()
    self.in_dim=input_dim
    self.layer=layer
    self.dim=dim
    self.dropout=dropout

    assert module in ['LSTM','GRU'],NotImplementedError
    self.hidden_state=None
    self.enable_cell=module=='LSTM'

    self.layers=getattr(nn.module)(input_dim,dim, num_layers=layer, dropout=dropout,batch_first=True)
    self.char_trans=nn.Linear(dim,vocab_size)
    self.final_dropout=nn.Dropout(dropout)

  def init_state(self,bs):
    device=next(self.parameters()).device
    if self.enable_cell:
      self.hidden_state=(torch.zeros((self.layer,bs,self.dim),device=device),
                         torch.zeros((self.layer,bs,self.dim),device=device))
    else:
      self.hidden_state=torch.zeros((self.layer,bs,self.dim),device=device)
    return self.get_state()

  def set_state(self, hidden_state):
    device=next(self.parameters()).device
    if enable_cell:
      self.hidden_state=(hidden_state[0].to(device),hidden_state[1].to(device))
    else:
      self.hidden_state=hidden_state.to(device)

  def get_state(self):
    ''' Return all hidden states/cells, for decoding purpose'''
    if self.enable_cell:
      return (self.hidden_state[0].cpu(), self.hidden_state[1].cpu())
    else:
      return self.hidden_state.cpu()

  def get_query(self):
    ''' Return state of all layers as query for attention '''
    if self.enable_cell:
      return self.hidden_state[0].transpose(0, 1).reshape(-1, self.dim*self.layer)
    else:
      return self.hidden_state.transpose(0, 1).reshape(-1, self.dim*self.layer)


  def forward(self,x):
    if not self.training:
      self.layers.flatten_parameters()
    x,self.hidden_state=self.layers(x.unsqueeze(1),self.hidden_state)
    x=x.squeeze(1)
    char=self.char_trans(self.final_dropout(x))
    return char,x


In [22]:
class Attention(nn.Module):
  def __init__(self,v_dim,q_dim,mode,dim,num_head, temperature, v_proj,loc_kernel_size,loc_kernel_num):
    super(Attention,self).__init__()

    self.v_dim=v_dim
    self.q_dim=q_dim
    self.mode=mode.lower()
    self.num_head=num_head

    self.proj_q=nn.Linear(q_dim, dim*num_head)
    self.proj_k=nn.Linear(v_dim,dim*num_head)
    self.v_proj=v_proj
    if v_proj:
      self.proj_v=nn.Linear(v_dim,dim*num_head)

    if self.mode=='dot':
      self.att_layer=ScaleDotAttention(temperature,self.num_head)
    elif self.mode=='loc':
      self.att_layer=LocationAwareAttention(
          loc_kernel_size,loc_kernel_num,dim,num_head,temperature)
    else:
      raise NotImplementedError

    if self.num_head>1:
      self.merge_head=nn.Linear(v_dim*num_head,v_dim)

    self.key=None
    self.value=None
    self.mask=None

  def reset_mem(self):
    self.key=None
    self.value=None
    self.mask=None
    self.att_layer.reset_mem()

  def set_mem(self,prev_attn):
    self.att_layer.set_mem(prev_attn)

  def forward(self,dec_state,enc_feat,enc_len):

    bs,ts,= enc_feat.shape
    query=torch.tanh(self.proj_q(dec_state))
    query=query.view(bs,self.num_head,self.dim).view(bs*self.num_head,self.dim)

    if self.key is None:
      self.att_layer.compute_mask(enc_feat,enc_len.to(enc_feat.device))

      self.key=torch.tanh(self.proj_k(enc_feat))
      self.valule=torch.tanh(self.proj_v(enc_feat)) if self.v_proj else enc_feat

      if self.num_head>1:
        self.key=self.key.view(bs,ts,self.num_head,self.dim).permute(0,2,1,3)
        self.key=self.key.contiguous().view(bs*self.num_head,ts,self.dim)
        if self.v_proj:
          self.value=self.value.view(bs,ts,self.num_head,self.v_dim).permute(0,2,1,3)
          self.value=self.value.contiguous().view(bs*self.num_head,ts,self.v_dim)
        else:
          self.value=self.value.repeat(self.num_head,1,1)

    context,attn=self.att_layer(query,self.key,self.value)
    if self.num_head>1:
      context=context.view(bs,self.num_head*self.v_dim)
      context=self.merge_head(context)

    return attn, context


In [23]:
class Encoder(nn.Module):
  def __init__(self,input_size,prenet,module, bidirection, dim, dropout,layer_norm,proj,sample_rate,sample_style):
    super(Encoder,self).__init__()

    self.vgg=prenet=='vgg'
    self.cnn=prenet=='cnn'
    self.sample_rate=1
    assert len(sample_rate)==len(dropout),'Number of layer mismatch'
    assert len(dropout)==len(dim),'Number of layer mismatch'
    num_layers=len(dim)
    assert num_layers>=1,'Encoder should have at least 1 layer'

    module_list=[]
    input_dim=input_size

    if self.vgg:
      vgg_extractor=VGGExtractor(input_size)
      module_list.append(vgg_extractor)
      input_dim=vgg_extractor.out_dim
      self.sample_rate=self.sample_rate*4

    if self.cnn:
      cnn_extractor=CNNExtractor(input_size,out_dim=dim[0])
      module_list.append(cnn_extractor)
      input_dim=cnn_extractor.out_dim
      self.sample_rate=self.sample_rate*4

    if module in ['LSTM','GRU']:
      for l in range(num_layers):
        module_list.append(RNNLayer(input_dim,module,dim[l],bidirection,dropout[l],layer_norm[l],sample_rate[l],sample_style,proj[l]))
        input_dim=module_list[-1].out_dim
        self.sample_rate=self.sample_rate*sample_rate[l]
    else:
      raise NonImplementedError

    self.in_dim=input_size
    self.out_dim=input_dim
    self.layers=nn.ModuleList(module_list)

  def forward(self,input_x,enc_len):
    for _, layer in enumerate(self.layers):
      input_x,enc_len=layer(input_x,enc_len)
    return input_x,enc_len

In [24]:
class LocationAwareAttention(BaseAttention):
  def __init__(temperature,num_head):
    self.prev_att=None
    #
    self.loc_conv=nn.Conv1d(num_head,kernel_num,kernel_size=2*kernel_size+1,padding=kernel_size, bias=False)
    self.gen_energy=nn.Linear(dim,1)
    self.dim=dim

  def reset_mem(self):
    super().reset_mem()
    self.prev_att=None

  def set_mem(self,prev_att):
    self.prev_att=prev_att


  # q=[batch_size, time, dim]
  # k=[batch_size, time, dim]
  # v=[batch_size, time, dim]
  def forward(self,q,k,v):
    bs_nj,ts,_=k.shape # [bs_nh, time, dim]
    bs=bs_nh//self.num_head

    if self.prev_att is None: #시작할때
      self.prev_att = torch.zeros((bs,self.num_head,ts)).to(k.device)
      for idx,sl in enumerate(self.k_len):
        self.prev_att[idx,:,:sl]=1.0/sl

    loc_content=torch.tanh(self.loc_proj(self.loc_conv(self.prev_att).transpose(1,2)))#local pattern 정보 학습
    loc_content=loc_content.unsqueeze(1).repeat(1,self.num_head,1,1).view(-1,ts,self.dim)
    q=q.unsqueeze(1) #[b,t,d]->[b,1,t,d]

    #[bs_nh,t,d] -> [bs_nh,t]
    energy=self.gen_energy(torch.tanh(k+q+loc_context)).squeeze(2)#t가 사라짐
    #[bs_nh,t] ->  [bs_nh,1, d]
    output,attn=self._attend(energy,v)# softmax로 attention weight로 만들고, weighted sum 진행
    attn=attn.view(bs,self.num_head,ts)
    #[bs_nh,t] -> [bs,num_head,t]
    self.prev_att=attn

    return output, attn

In [25]:
class RNNLM(nn.Module):

  def __init__(self,vocab_size,emb_typing,emb_dim,module,dim,n_layers,dropout):
    super().__init__()
    self.dim=dim
    self.n_layers=n_layers
    self.emb_tying=emb_tying
    if emb_tying:
      assert emb_dim==dim
    self.vocab_size=vocab_size
    self.emb=nn.Embedding(vocab_size,emb_dim)
    self.dp1=nn.Dropout(dropout)
    self.dp2=nn.Dropout(dropout)
    self.rnn=getattr(nn,module.upper())(
        emb_dim,dim,num_layers=n_layers,dropout=dropout,batch_first=True
    )
    if not self.emb_tying:
      self.trans=nn.Linear(dim,vocab_size)

  def create_msg(self):
    msg = ['Model spec.| RNNLM weight tying = {}, # of layers = {}, dim = {}'.format(
            self.emb_tying, self.n_layers, self.dim)]
    return msg

  def forward(self,x,lens,hidden=None):
    emb_x=self.dp1(self.emb(x))
    if not self.training:
      self.rnn.flatten_parameters()
    packed=nn.utils.rnn_pack_padded_sequence(emb_x,lens,batch_first=True,enforce_sorted=False)
    outputs,hidden=self.rnn(packed,hidden)
    outputs,_=nn.utils.rnn.pad_packed_sequence(
        outputs,batch_first=True)
    if self.emb_tying:
      outputs=F.linear(self.dp2(outputs),self.emb.weight)
    else:
      outputs=self.trans(self.dp2(outputs))
    return outputs, hidden

In [26]:
LOG_ZERO=-10000000.0

class CTCPrefixScore():
  def __init__(self,x):
    self.logzero = -100000000.0
    self.blank = 0
    self.eos = 1
    self.x = x.cpu().numpy()[0]
    self.odim = x.shape[-1]
    self.input_length = len(self.x)

  def init_state(self):
    r=np.full((self.input_length,2),self.logzero,dtype=np.float32)

    r[0,1]=self.x[0,self.blank]
    for i in range(1,self.input_length):
      r[i,1]=r[i-1,1]+self.x[i,self.blank]
    return r

  def full_compute(self,g,r_prev):
    prefix_length=len(g)
    last_char=g[-1] if prefix_length>0 else 0

    r = np.full((self.input_length, 2, self.odim),
                    self.logzero, dtype=np.float32) #차원, 채울 값(log 0)

    start=max(1,prefix_length)

    if prefix_length==0:
      r[0,0,:] = self.x[0,:]

    psi = r[start-1, 0, :]

    phi = np.logaddexp(r_prev[:, 0], r_prev[:, 1])

    for t in range(start,self.input_length):
      prev_blank=np.full((self.odim),r_prev[t-1,1],dtype=np.float32) #시점 t에서 blank 출력한 경우
      prev_nonblank=np.full((self.odim),r_prev[t-1,0],dtype=np.float32) #시점 t에서 문자 출력한 경우
      prev_nonblank[last_char]=self.logzero

      phi = np.logaddexp(prev_nonblank, prev_blank)
      # P(h|current step is non-blank) = [ P(prev. step = y) + P()]*P(c)
      r[t, 0, :] = np.logaddexp(r[t-1, 0, :], phi) + self.x[t, :]
      # P(h|current step is blank) = [P(prev. step is blank) + P(prev. step is non-blank)]*P(now=blank)
      r[t, 1, :] = np.logaddexp(
          r[t-1, 1, :], r[t-1, 0, :]) + self.x[t, self.blank]
      psi = np.logaddexp(psi, phi+self.x[t, :])

    return psi,np.rollaxis(r,2)

  def cheap_compute(self, g, r_prev, candidates):
    '''Given prefix g, return the probability of all possible sequence y (where y = concat(g,c))
        This function considers only those tokens in candidates for c (memory efficient)'''
    prefix_length = len(g)
    odim = len(candidates)
    last_char = g[-1] if prefix_length > 0 else 0

    # init. r
    r = np.full((self.input_length, 2, len(candidates)),
                self.logzero, dtype=np.float32)

    # start from len(g) because is impossible for CTC to generate |y|>|X|
    start = max(1, prefix_length)

    if prefix_length == 0:
        r[0, 0, :] = self.x[0, candidates]    # if g = <sos>

    psi = r[start-1, 0, :]
    # Phi = (prev_nonblank,prev_blank)
    sum_prev = np.logaddexp(r_prev[:, 0], r_prev[:, 1])
    phi = np.repeat(sum_prev[..., None],odim,axis=-1)
    # Handle edge case : last tok of prefix in candidates
    if  prefix_length>0 and last_char in candidates:
        phi[:,candidates.index(last_char)] = r_prev[:,1]

    for t in range(start, self.input_length):
        # prev_blank
        # prev_blank = np.full((odim), r_prev[t-1, 1], dtype=np.float32)
        # prev_nonblank
        # prev_nonblank = np.full((odim), r_prev[t-1, 0], dtype=np.float32)
        # phi = np.logaddexp(prev_nonblank, prev_blank)
        # P(h|current step is non-blank) =  P(prev. step = y)*P(c)
        r[t, 0, :] = np.logaddexp( r[t-1, 0, :], phi[t-1]) + self.x[t, candidates]
        # P(h|current step is blank) = [P(prev. step is blank) + P(prev. step is non-blank)]*P(now=blank)
        r[t, 1, :] = np.logaddexp( r[t-1, 1, :], r[t-1, 0, :]) + self.x[t, self.blank]
        psi = np.logaddexp(psi, phi[t-1,]+self.x[t, candidates])

    # P(end of sentence) = P(g)
    if self.eos in candidates:
        psi[candidates.index(self.eos)] = sum_prev[-1]
    return psi, np.rollaxis(r, 2)



In [27]:
class CTCHypothesis(): #ctc 가설 지정, 두 종류의 확률을 따로 관리함
  def __init__(self):
    self.y=[] #
    self.Pr_y_t_blank=0.0 # 마지막 토큰이 BLANK
    self.Pr_y_t_nblank=LOG_ZERO #마지막 토큰이 글자일 확률

    self.Pr_y_t_blank_bkup=0.0 # T-1 시점의 확률 보존용
    self.Pr_y_t_nblank_bkup=LOG_ZERO

    self.lm_output=None
    self.lm_hidden=None
    self.updated_lm=False

  def update_lm(self,outpu,hidden): #현재 가설에 대한 LM 모델 결과 저장
    self.lm_output=output
    self.lm_hidden=hidden
    self.updated_lm=True

  def get_len(self):
    return len(self.y)

  def get_string(self):
    return ''.join([str(s) for s in self.y])

  def get_score(self): #현재시점 T까지의 total log-prob 계산
    return np.logaddexp(self.Pr_y_t_blank,self.Pr_y_t_nblank)

  def get_final_score(self): #최종 스코어를 출력 길이로 정규화 한 점수
    if len(self.y)>0:
      return np.logaddexp(self.Pr_y_t_blank_bkup,self.Pr_y_t_nblank_bkup)
    else:
      return self.Pr_y_t_blank_bkup

  def check_same(self,y_2): #가설 y와 y2가 완전히 같은지 확인, 중복가설 제거
    if len(self.y)!=len(y_2):
      return False
    for i in range(len(self.y)):
      if self.y[i]!=y_2[i]:
        return False
    return True

  def update_Pr_nblank(self, ctc_y_t): #최종 마지막 토큰이 non-blank 일 확률 업데이트
        # ctc_y_t  : Pr(ye,t|x)
        # Pr+(y,t) = Pr+(y,t-1) * Pr(ye,t|x)
        self.Pr_y_t_nblank += ctc_y_t

  def update_Pr_nblank_prefix(self, ctc_y_t, Pr_y_t_blank_prefix, Pr_y_t_nblank_prefix, Pr_ye_y=None):
    # ctc_y_t  : Pr(ye,t|x)
    lm_prob = Pr_ye_y if Pr_ye_y is not None else 0.0
    if len(self.y) == 0: return
    if len(self.y) == 1:
        Pr_ye_y_prefix = ctc_y_t + lm_prob + np.logaddexp(Pr_y_t_blank_prefix, Pr_y_t_nblank_prefix)
    else:
        # Pr_ye_y : LM Pr(ye|y)
        Pr_ye_y_prefix = ctc_y_t + lm_prob + (Pr_y_t_blank_prefix if self.y[-1] == self.y[-2] \
                                    else np.logaddexp(Pr_y_t_blank_prefix, Pr_y_t_nblank_prefix))
    # Pr+(y,t) = Pr+(y,t) + Pr(ye,y^,t)
    self.Pr_y_t_nblank = np.logaddexp(self.Pr_y_t_nblank, Pr_ye_y_prefix)

  def update_Pr_blank(self, ctc_blank_t):
    # Pr-(y,t) = Pr(y,t-1) * Pr(-,t|x)
    self.Pr_y_t_blank = np.logaddexp(self.Pr_y_t_nblank_bkup, self.Pr_y_t_blank_bkup) + ctc_blank_t

  def add_token(self,token,):
    lm_prob=Pr_k_y if Pr_k_y is not None else 0.0
    if len(self.y) == 0:
      Pr_y_t_nblank_new = ctc_token_t + lm_prob + np.logaddexp(self.Pr_y_t_blank_bkup, self.Pr_y_t_nblank_bkup)
    else:
      # Pr_k_y : LM Pr(k|y)
      Pr_y_t_nblank_new = ctc_token_t + lm_prob + (self.Pr_y_t_blank_bkup if self.y[-1] == token else \
                                    np.logaddexp(self.Pr_y_t_blank_bkup, self.Pr_y_t_nblank_bkup))
    self.Pr_y_t_blank  = LOG_ZERO
    self.Pr_y_t_nblank = Pr_y_t_nblank_new

    self.Pr_y_t_blank_bkup  = self.Pr_y_t_blank
    self.Pr_y_t_nblank_bkup = self.Pr_y_t_nblank

    self.y.append(token)

  def orig_backup(self):
    self.Pr_y_t_blank_bkup  = self.Pr_y_t_blank
    self.Pr_y_t_nblank_bkup = self.Pr_y_t_nblank

In [28]:
class CTCBeamDecoder(nn.Module):
  def __init__(self,asr,vocab_range,beam_size,vocab_candidate,lm_path='',lm_config='',lm_weight=0.0,device=None):
    super().__init__()
    self.asr=asr #ASR 모델
    self.vocab_range=vocab_range
    self.beam_size=beam_size
    self.vocab_candidate=vocab_candidate
    assert self.vocab_cand<=len(self.vocab_range)
    assert self.asr.enable_ctc

    self.apply_lm_weight>0
    self.lm_w=0
    if self.apply_lm: #디코더 값으로 학습 중간 확인
      self.device=device
      self.lm_w=lm_weight #LM의 가중치 비율
      self.lm_path=lm_path
      lm_config=yaml.load(open(lm_config,'r'),Loader=yaml.FullLoader)
      self.lm=RMMLM(self.asr.vocab_size,**lm_config['model']).to(self.device)#CTC 토큰 예측기
      self.lm.load_state_dict(torch.load(
          self.lm_path,map_location='cpu')['model'])

      self.lm.eval()

  def create_msg(self):
    msg = ['Decode spec| CTC decoding \t| Beam size = {} \t| LM weight = {}'.format(self.beam_size, self.lm_w)]
    return msg

  def forward(self,feat,feat_len):
    assert feat.shape[0] == 1, "Batchsize == 1 is required for beam search"

    ctc_output,encode_len,att_output,att_align,dec_state=self.asr(feat,feat_len,10)
    del encode_len, att_output,att_align,dec_state,feat_len
    ctc_output=F.log_softmax(ctc_output[0],dim=-1).cpu().detach().numpy()
    T=len(ctc_output) #CTC의 전체 길이

    B=[CTCHypothesis()]
    if self.apply_lm: #LM 적용 여부
    #LM 입력값 , TOKEN(B,T), LENGTH(T), HIDDEN_STATE(NONE)
    #출력값(B,T,V)
      output,hidden=self.lm(torch.zeros((1,1),dtype=torch.long).to(self.device),torch.ones(1,dtype=torch.long).to(self.device),None)
      B[0].update_lm((output).log_softmax(dim=-1).squeeze().cpu().numpy(),hidden)

    start=True
    for t in range(T):# 각 시간별
      if np.argmax(ctc_output[t])==0 and start: #PAD 토큰이 최빈값이면 해당 시점 무시
        continue
      else:
        start=False
      B_new=[]

      for i in range(len(B)): #배치 사이즈
        B_i_new=copy.deepcopy(B[i]) #새로운 가설을 만들기 위한 복사본
        if B_i_new.get_len()>0:
          if B_i_new.y[-1]==1:#현재 가설이 끝났다면(예측이 전부 다 됐다면)
            B_new.append(B_i_new) #끝났으면 후보에다가 추가
            continue
          B_i_new.update_Pr_nblank(ctc_output[t,B_i_new.y[-1]])#nblank 확률 업데이트

          for j in range(len(B)): #prefix 중복 가설 처리
            if i!=j and B[j].check_same(B_i_new.y[:-1]):
              lm_prob=0.0
              if self.apply_lm:
                lm_prob=self.lm_w*B[j].lm_output[B_i_new.y[-1]]
              B_i_new.update_Pr_nblank_prefix(ctc_output[t,B_i_new.y[-1]],#nblank 확률 업데이트
                                              B[j].Pr_y_t_blank,
                                              B[j].Pr_y_t_nblank,lm_prob)
              break

      B_i_new.update_Pr_blank(ctc_output[t,0]) #blank 확률 업데이트

      if self.apply_lm:
        lm_hidden=B_i_new.lm_hidden
        lm_probs=B_i_new.lm_output
      else:
        lm_hidden=None
        lm_probs=None

      if self.apply_lm: #lm이 적용되는 경우 후보를 정렬, soft fusion 방식
        ctc_vocab_cand=sorted(zip(
            self.vocab_range,ctc_output[t,self.vocab_ragne]+self.lm_w*lm_probs[self.vocab_range]),
                              reverse=True,key=lambda x:x[1])
      else: #다른 경우에 pure ctc 확률로 정렬
        ctc_vocab_cand=sorted(zip(self.vocab_range,ctc_output[t,self.vocab_range]),reverse=True,key=lambda x:x[1])

      for j in range(self.vocab_cand): # 각 후보별 내용 정리
        k=ctc_vocab_cand[j][0]
        hyp_yk=copy.deepcopy(B_i_new)
        lm_prob=0.0 if not self.apply_lm else self.lm_w*lm_probs[k]
        hyp_yk.add_token(k,ctc_output[t,kl],lm_prob)
        hyp_yk.updated_lm=False #lm 업데이트 여부
        B_new.append(hyp_yk)# 가설 추가
      B_i_new.orig_backup()
      B_i_new.append(B_i_new)
    del B
    B=[]

    B_new=sorted(B_new,key=lambda x:x.get_string()) #가설 정리
    B.append(B_new[0])
    for i in range(1,len(B_new)): #중복 가설 제거
      if B_new[i].check_same(B[-1].y): #동일한 경우
        if B_new[i].get_score()>B[-1].get_score():#더 가능성 높은 녀석을 추가
          B[-1]=B_new[i]
        continue
      else:
        B.append(B_new[i])
    del B_new

    if t==T-1:#마지막 시점에서는 정규화된 점수 사용
      B=sorted(B,reverse=True,key=lambda x:x.get_final_score())
    else:#중간 시점에서는 누적점수 사용
      B=sorted(B,reverse=True,key=lambda x:x.get_score())
    if len(B)>self.beam_size:#beam 갯수가 적으면 적은만큼만 사용
      B=B[:self.beam_size]

    if self.apply_lm and t<T-1:#lm 상태 업데이트
      for i in range(len(B)): #현재 가설의 마지막 토큰을 넣고 다음 토큰을 예측
        if B[i].get_len()>0 and not B[i].updated_lm:
          output,hidden=self.lm(B[i].y[-1]*torch.ones((1,1),dtype=torch.long).to(self.device),torch.ones(1,dtype=torch.long).to(self.device),B[i].lm_hidden)
          B[i].update_lm((output).log_softmax(dim=-1).squeeze().cpu().numpy(),hidden)

    return [b.y for b in B]

In [29]:

CTC_BEAM_RATIO = 1.5   # DO NOT CHANGE THIS, MAY CAUSE OOM


class BeamDecoder(nn.Module): #빔서치 디코더
    ''' Beam decoder for ASR '''

    def __init__(self, asr, emb_decoder, beam_size, min_len_ratio, max_len_ratio,
                 lm_path='', lm_config='', lm_weight=0.0, ctc_weight=0.0):
        super().__init__()
        # Setup
        self.beam_size = beam_size #몇개의 빔 사용할지
        self.min_len_ratio = min_len_ratio #최소, 최대 출력 길이 비율
        self.max_len_ratio = max_len_ratio
        self.asr = asr

        # ToDo : implement pure ctc decode
        assert self.asr.enable_att

        # Additional decoding modules
        self.apply_ctc = ctc_weight > 0 #ctc 적용 비율
        if self.apply_ctc: #ctc 사용하는 경우
            assert self.asr.ctc_weight > 0, 'ASR was not trained with CTC decoder'
            self.ctc_w = ctc_weight #비율을 적용하고
            self.ctc_beam_size = int(CTC_BEAM_RATIO * self.beam_size)

        self.apply_lm = lm_weight > 0 #lm 적용 비율
        if self.apply_lm: #lm을 사용하는 경우
            self.lm_w = lm_weight
            self.lm_path = lm_path
            lm_config = yaml.load(open(lm_config, 'r'), Loader=yaml.FullLoader)
            self.lm = RNNLM(self.asr.vocab_size, **lm_config['model']) #rnnlm 모델 설정 가져오기
            self.lm.load_state_dict(torch.load(
                self.lm_path, map_location='cpu')['model'])
            self.lm.eval()

        self.apply_emb = emb_decoder is not None #임베딩 적용하는 경우
        if self.apply_emb:
            self.emb_decoder = emb_decoder

    def create_msg(self):
        msg = ['Decode spec| Beam size = {}\t| Min/Max len ratio = {}/{}'.format(
            self.beam_size, self.min_len_ratio, self.max_len_ratio)]
        if self.apply_ctc:
            msg.append(
                '           |Joint CTC decoding enabled \t| weight = {:.2f}\t'.format(self.ctc_w))
        if self.apply_lm:
            msg.append('           |Joint LM decoding enabled \t| weight = {:.2f}\t| src = {}'.format(
                self.lm_w, self.lm_path))
        if self.apply_emb:
            msg.append('           |Joint Emb. decoding enabled \t| weight = {:.2f}'.format(
                self.lm_w, self.emb_decoder.fuse_lambda.mean().cpu().item()))

        return msg

    def forward(self, audio_feature, feature_len): #작동 순서
        # Init.
        #빔서치 위함 왜?
        assert audio_feature.shape[0] == 1, "Batchsize == 1 is required for beam search"

        batch_size = audio_feature.shape[0]# 오디오 피쳐 첫번째 값, batch_size
        device = audio_feature.device
        dec_state = self.asr.decoder.init_state(
            batch_size)                           # Init zero states
        self.asr.attention.reset_mem()      #초기화      # Flush attention mem
        # Max output len set w/ hyper param.
        max_output_len = int(
            np.ceil(feature_len.cpu().item()*self.max_len_ratio)) #출력 최대 비율 가져오기, 하이퍼파라미터임
        # Min output len set w/ hyper param.
        min_output_len = int(
            np.ceil(feature_len.cpu().item()*self.min_len_ratio))
        # Store attention map if location-aware
        store_att = self.asr.attention.mode == 'loc'
        prev_token = torch.zeros( #시작 첫 토큰, [1,1]
            (batch_size, 1), dtype=torch.long, device=device)     # Start w/ <sos>
        # Cache of beam search
        final_hypothesis, next_top_hypothesis = [], [] #최종 결과 후보들, 현재 스텝에서 상위로 넘어갈 beam size 개의 후보들
        # Incase ctc is disabled
        ctc_state, ctc_prob, candidates, lm_state = None, None, None, None
        #prefix 계산시 필요한 상태, 이전까지의 ctc 누적 확률, 현재 고려중인 top-k 후보 토큰들, rnn기반 hidden state

        # Encode, 인코딩을 통한 mel-spectrogram -> latent_representation으ㅜ로 변경
        encode_feature, encode_len = self.asr.encoder(
            audio_feature, feature_len)

        # CTC decoding, 디코딩 적용
        # encode_feature =[B,T_ENC, HIDDEN_SIZE], H_ENC=VOCAB_SIZE(사용 가능한 단어들)
        # output_feature =[B,T,VOCAB_SIZE]
        # 소프트 맥스 적용 = [B,T,VOCAB_SIZE]-> VOCAB_SIZE중 1개만 1로 변경됨
        if self.apply_ctc:
            ctc_output = F.log_softmax( #
                self.asr.ctc_layer(encode_feature), dim=-1)
            ctc_prefix = CTCPrefixScore(ctc_output)
            ctc_state = ctc_prefix.init_state()

        # Start w/ empty hypothesis, 시작지점 초기화, ctc_state=0인 지점은 확률이 0인 지점
        prev_top_hypothesis = [Hypothesis(decoder_state=dec_state, output_seq=[],
                                          output_scores=[], lm_state=None, ctc_prob=0,
                                          ctc_state=ctc_state, att_map=None)]
        # Attention decoding
        for t in range(max_output_len):# 최대 길이까지 계산 시작
            for hypothesis in prev_top_hypothesis: #방금 전까지의 결과물들에 대한 방식 계산
                # Resume previous step
                prev_token, prev_dec_state, prev_attn, prev_lm_state, prev_ctc_state = hypothesis.get_state(
                    device)# 각 가설에 대한 내용들을 기반으로 지정
                self.asr.set_state(prev_dec_state, prev_attn) #

                # Normal asr forward
                attn, context = self.asr.attention(
                    self.asr.decoder.get_query(), encode_feature, encode_len)
                asr_prev_token = self.asr.pre_embed(prev_token)
                decoder_input = torch.cat([asr_prev_token, context], dim=-1)
                cur_prob, d_state = self.asr.decoder(decoder_input)


                # Embedding fusion (output shape 1xV)
                if self.apply_emb: #임베딩 적용시
                    _, cur_prob = self.emb_decoder( d_state, cur_prob, return_loss=False)
                else:
                    cur_prob = F.log_softmax(cur_prob, dim=-1)

                # Perform CTC prefix scoring on limited candidates (else OOM easily)
                # ctc score는 어떤 prefix가 있을때, 거기서 시작하는 모든 ctc path의 log prob 합
                #
                if self.apply_ctc: #ctc 적용하는 경우
                    # TODO : Check the performance drop for computing part of candidates only
                    # cur_prob= 현재 log prob[1,vocab_size]의 log prob
                    # vocab 중에서 상위 k개(beam size)갯수 만큼 후보군 뽑기
                    _, ctc_candidates = cur_prob.squeeze(0).topk(self.ctc_beam_size, dim=-1)
                    candidates = ctc_candidates.cpu().tolist()# 리스트화
                    #ctc_prob=[후보군 갯수]
                    #ctc_state=[각 후보군의 ctc log prob]
                    ctc_prob, ctc_state = ctc_prefix.cheap_compute( # ctc prefix score 계산
                        hypothesis.outIndex, prev_ctc_state, candidates)
                    #
                    # TODO : study why ctc_char (slightly) > 0 sometimes
                    ctc_char = torch.FloatTensor(ctc_prob - hypothesis.ctc_prob).to(device)
                    # 현재 경로의 확률 상에서 각각의 후보가 얼마만큼의 기여를 했는지 계산

                    # Combine CTC score and Attention score (HACK: focus on candidates, block others)
                    # hack_ctc_char=[1,vocab_size]를 만들고 log inf로 채움 -> 확률 0
                    hack_ctc_char = torch.zeros_like(cur_prob).data.fill_(LOG_ZERO)
                    #현재 경로에서의 각 후보를 가지고 계산 시작
                    for idx, char in enumerate(candidates): #ctc 각 순서, 문자(id값)
                        hack_ctc_char[0, char] = ctc_char[idx]
                    # attention decoder가 예측한 다음 토큰의 log-prob - [1,vocab_size]
                    # ctc 보정 점수(후보 토큰만 값이 있고, 나머진 log0)- [1,vocab_size]
                    cur_prob = (1-self.ctc_w)*cur_prob + self.ctc_w*hack_ctc_char  # ctc_char
                    cur_prob[0, 0] = LOG_ZERO  # Hack to ignore <sos>
                    # 다음 상태에 대한 결과를 확인함

                # Joint RNN-LM decoding
                if self.apply_lm: #lm 모델을 사용하는 경우
                    # assuming batch size always 1, resulting 1x1
                    lm_input = prev_token.unsqueeze(1) #[1,1]
                    lm_output, lm_state = self.lm(
                        lm_input, torch.ones([batch_size]), hidden=prev_lm_state)
                    # assuming batch size always 1,  resulting 1xV
                    lm_output = lm_output.squeeze(0) #[1,1,vocab_size]
                    cur_prob += self.lm_w*lm_output.log_softmax(dim=-1)
                    #cur_prob는 [1,vocab_size]

                #
                # Beam search
                # Note: Ignored batch dim.
                topv, topi = cur_prob.squeeze(0).topk(self.beam_size) #후보군 선택
                # prev_attn = location 관련된 경우, 사용할 수 있도록 만듦
                prev_attn = self.asr.attention.att_layer.prev_att.cpu() if store_att else None
                #final은 완성된 시퀀스, top은 확장 가능한 경로 리스트
                final, top = hypothesis.addTopk(topi, topv, self.asr.decoder.get_state(), att_map=prev_attn,
                                                lm_state=lm_state, ctc_state=ctc_state, ctc_prob=ctc_prob,
                                                ctc_candidates=candidates)
                # Move complete hyps. out
                if final is not None and (t >= min_output_len):#끝나지 않은 경우, 다음으로 넘길 값으로 추가함
                    final_hypothesis.append(final)
                    if self.beam_size == 1:
                        return final_hypothesis
                next_top_hypothesis.extend(top)

            # Sort for top N beams
            next_top_hypothesis.sort(key=lambda o: o.avgScore(), reverse=True)# 각 경로의 최댓값 k개 정리
            prev_top_hypothesis = next_top_hypothesis[:self.beam_size]
            next_top_hypothesis = []

        # Rescore all hyp (finished/unfinished)
        final_hypothesis += prev_top_hypothesis
        final_hypothesis.sort(key=lambda o: o.avgScore(), reverse=True)

        return final_hypothesis[:self.beam_size]



In [30]:
import torch
import numpy as np
from functools import partial

class Optimizer():
    def __init__(self, parameters, optimizer, lr, eps, lr_scheduler, tf_start=1, tf_end=1, tf_step=1, **kwargs):

        # Setup teacher forcing scheduler
        self.tf_type = tf_end != 1
        self.tf_rate = lambda step: max(
            tf_end, tf_start-(tf_start-tf_end)*step/tf_step)

        # Setup torch optimizer
        self.opt_type = optimizer
        self.init_lr = lr
        self.sch_type = lr_scheduler
        opt = getattr(torch.optim, optimizer)
        if lr_scheduler == 'warmup':
            warmup_step = 4000.0
            init_lr = lr
            self.lr_scheduler = lambda step: init_lr * warmup_step ** 0.5 * \
                np.minimum((step+1)*warmup_step**-1.5, (step+1)**-0.5)
            self.opt = opt(parameters, lr=1.0)
        elif lr_scheduler == 'spec-aug-basic':
            # Scheduler from https://arxiv.org/pdf/1904.08779.pdf
            self.lr_scheduler = partial(speech_aug_scheduler, s_r=500,
                                        s_i=20000, s_f=80000, peak_lr=lr)
            self.opt = opt(parameters, lr=lr, eps=eps)
        elif lr_scheduler == 'spec-aug-double':
            # Scheduler from https://arxiv.org/pdf/1904.08779.pdf
            self.lr_scheduler = partial(speech_aug_scheduler, s_r=1000,
                                        s_i=40000, s_f=160000, peak_lr=lr)
            self.opt = opt(parameters, lr=lr, eps=eps)
        else:
            self.lr_scheduler = None
            self.opt = opt(parameters, lr=lr, eps=eps)  # ToDo: 1e-8 better?

    def get_opt_state_dict(self):
        return self.opt.state_dict()

    def load_opt_state_dict(self, state_dict):
        self.opt.load_state_dict(state_dict)

    def pre_step(self, step):
        if self.lr_scheduler is not None:
            cur_lr = self.lr_scheduler(step)
            for param_group in self.opt.param_groups:
                param_group['lr'] = cur_lr
        self.opt.zero_grad()
        return self.tf_rate(step)

    def step(self):
        self.opt.step()

    def create_msg(self):
        return ['Optim.spec.| Algo. = {}\t| Lr = {}\t (Scheduler = {})| Scheduled sampling = {}'
                .format(self.opt_type, self.init_lr, self.sch_type, self.tf_type)]

def speech_aug_scheduler(step, s_r, s_i, s_f, peak_lr):
    # Starting from 0, ramp-up to set LR and  converge to 0.01*LR, w/ exp. decay
    final_lr_ratio = 0.01
    exp_decay_lambda = -np.log10(final_lr_ratio)/(s_f-s_i) # Approx. w/ 10-based
    cur_step = step+1

    if cur_step<s_r:
        # Ramp-up
        return peak_lr*float(cur_step)/s_r
    elif cur_step<s_i:
        # Hold
        return peak_lr
    elif cur_step<=s_f:
        # Decay
        return peak_lr*np.power(10,-exp_decay_lambda*(cur_step-s_i))
    else:
        # Converge
        return peak_lr*final_lr_ratio

In [31]:
class BertLikeSentencePieceTextEncoder(object):
  def __init__(self,text_encoder):
    if not isinstance(text_encoder, text.SubwordTextEncoder):
      raise TypeError(
                "`text_encoder` must be an instance of `src.text.SubwordTextEncoder`.")
    self.text_encoder=text_encoder

  def vocab_size(self):
    return self.text_encoder.vocab_size+3

  def cls_idx(self):
    return self.vocab_size-3

  def mask_idx(self):
    return self.vocab_size-2

  def eos_idx(self):
    return self.text_encoder.eos_idx

  def generate_embedding(bert_model,labels):
    batch_size,time=labels.shape

    cls_ids=torch.full((batch_size,1),bert_model.bert_text_encoder.cls_ics,dtype=labels.dtype,device=labels.device)
    bert_labels=torch.cat([cls_ids,labels],1)
    eos_idx=bert_model.bert_text_encoder.eos_idx
    sep_idx=bert_model.bert_text_encoder.sep_idx
    bert_labels[bert_labels==eos_idx]=sep_idx

    embedding,_=bert_model(bert_labes,output_all_encoded_layers=True)

    embedding = torch.stack(embedding).sum(0)
    embedding=embedding[:,1:]

    assert labels.shape==embedding.shape[:-1]

    return embedding


  def load_fine_tuned_model(bert_model, text_encoder, path):
    """Load fine-tuned bert model given text encoder and checkpoint path."""
    bert_text_encoder = BertLikeSentencePieceTextEncoder(text_encoder)

    model = BertForMaskedLM.from_pretrained(bert_model)
    model.bert_text_encoder = bert_text_encoder
    model.bert.embeddings.word_embeddings = nn.Embedding(
        bert_text_encoder.vocab_size, model.bert.embeddings.word_embeddings.weight.shape[1])
    model.config.vocab_size = bert_text_encoder.vocab_size
    model.cls = BertOnlyMLMHead(
        model.config, model.bert.embeddings.word_embeddings.weight)

    model.load_state_dict(torch.load(path))

    return model



In [32]:
class BertEmbeddingPredictor(nn.Module):
  def __init__(self,bert_model,text_encoder,path):
    super(BertEmbeddingPredictor,self).__init__()
    self.modle=load_fine_tuned_model(bert_model,text_encoder,path)

  def forward(self,labels):
    self.eval()
    return generate_embedding(self.model,labels)

In [33]:
class EmbeddingRegularizer(nn.Module):
  def __init__(self,tokenizer,dec_dim,enable,src,distance,weight,fuse,temperature,freeze=True,fuse_normalize=False,dropout=0.0, bert=None):
    super(EmbeddingRegularizer,self).__init__()
    self.enable=enable
    if enable:
      if bert is not None:
        self.use_bert=True
        if not isinstance(bert,str):
          raise ValueError(
                        "`bert` should be a str specifying bert config such as \"bert-base-uncased\".")
        self.emb_table=BertEmbeddingPredictor(bet,tokenizer,src)
        vocab_size,emb_dim=self.emb_table.model.embeddings.word_embeddings.weight.shape
        vocab_size=vocab_size-3
        self.dim=emb_dim
      else:
        self.use_bert=False
        pretrained_emb=torch.FloatTensor(load_embedding(tokenizer,src))
        vocab_size=emb_dim=pretrained_emb.shape
        self.dim=emb_dim

        self.emb_table=nn.Embedding.from_pretrained(pretrained_emb,freeze=freeze,padding_idx=0)

        self.emb_net=nn.Sequential(nn.Linear(dec_dim,(emb_dim+dec_dim)//2),
                                   nn.ReLU(),
                                   nn.Linear((emb_dim+dec_dim)//2,emb_dim))

        self.weight=weight
        self.distance=distance
        self.fuse_normalize=fuse_normalize
        if distance=='CosEmb':
          self.measurement=nn.CosineEmbeddingLoss(reduction='none')
        elif distance=='MSE':
          self.measurement=nn.MSELoss(reduction='none')
        else:
          raise NotImplementedError

        self.apply_dropout=dropout>0
        if self.apply_dropout:
          self.dropout=nn.Dropout(dropout)

        self.apply_fuse=fuse!=0
        if self.apply_fuse:
          if fuse==-1:
            self.fuse_type="learnable"
            self.fuse_learnable=True
            self.fuse_lambda=nn.Parameter(data=torch.FloatTensor([0.5]))
          elif fuse==-2:
            self.fuse_type="vocab_wise learnable"
            self.fuse_learnable=True
            self.fuse_lambda=nn.Parameter(data=torch.ones((vocab_size))*0.5)
          else:
            self.fuse_type=str(fuse)
            self.fuse_learnable=False
            self.register_buffer('fuse_lambda',torch.FloatTensor([fuse]))

          if temperature==-1:
            self.temperature="learnable"
            self.temp=nn.Parameter(data=torch.FloatTensor([1]))
          elif temperature==-2:
            self.temperature="elementwise"
            self.temp=nn.Parameter(torch.ones((vocab_size)))
          else:
            self.temperature=str(temperature)
            self.register_buffer('temp',torch.FloatTensor([temperature]))
          self.eps=1e-8

  def create_msg(self):
    msg = ['Plugin.    | Word embedding regularization enabled (type:{}, weight:{})'.format(
        self.distance, self.weight)]
    if self.apply_fuse:
        msg.append('           | Embedding-fusion decoder enabled ( temp. = {}, lambda = {} )'.
                    format(self.temperature, self.fuse_type))
    return msg

  def get_weight(self):
    if self.fuse_learnable:
      return torch.sigmoid(self.fuse_lambda).mean().cpu().data
    else:
      return self.fuse_lambda

  def get_temp(self):
    return nn.functional.relu(self.temp).mean()

  def fuse_prob(self,x_emb,dec_logit):
    if self.fuse_normalize:
      emb_logit=nn.functional.linear(nn.functional.normalize(x_emb,dim=-1),
                                     nn.functional.normalize(self.emb_table.weight,dim=-1))
    else:
      emb_logit=nn.functional.linear(x_emb,self.emb_table.weight)
    emb_prob=(nn.functional.relu(self.temp)*emb_logit).softmax(dim=-1)
    dec_prob=dec_logit.softmax(dim=-1)

    if self.fulse_learnable:
      fused_prob=(1-torch.sigmoid(self.fuse_lambda))*dec_prob+torch.sigmoid(self.fuse_lambda)*emb_prob
    else:
      fused_prob=(1-self.fuse_lambda)*dec_prob+self.fuse_lambda*emb_prob
    log_fused_prob=(fused_prob+self.eps).log()

    return log_fused_prob

  def forward(self,dec_state,dec_logit,label=None,return_loss=True):
    log_fused_prob=None
    loss=None

    if self.apply_dropout:
      dec_state=self.dropout(dec_state)
    x_emb=self.emb_net(dec_state)

    if return_loss:
      b,t=label.shape
      if self.use_bert:
        with torch.no_grad():
          y_emb=self.emb_table(label).contiguous()
      else:
        y_emb=self.emb_table(label)

      if self.distance=='CosEmb':
        loss=self.measurement(x_emb.view(-1,self.dim),y_emb.view(-1,self.dim),torch.ones(1).to(dec_state.device))
      else:
        loss=self.measurement(x_emb.view(-1,self.dim),y_emb.view(-1,self.dim))
      loss=loss.view(b,t)
      loss=torch.where(label!=0, loss, torch.zeros_like(loss))
      loss=torch.mean(loss.sum(dim=-1)/(label!=0).sum(dim=-1).float())

    if self.apply_fuse:
      log_fused_prob=self.fuse_prob(x_emb,dec_logit)

    return loss, log_fused_prob

In [34]:
default_hparas = {
    'GRAD_CLIP': 5.0,          # Grad. clip threshold
    'PROGRESS_STEP': 100,      # Std. output refresh freq.
    # Decode steps for objective validation (step = ratio*input_txt_len)
    'DEV_STEP_RATIO': 1.2,
    # Number of examples (alignment/text) to show in tensorboard
    'DEV_N_EXAMPLE': 4,
    'TB_FLUSH_FREQ': 180       # Update frequency of tensorboard (secs)
}

In [35]:
class BaseSolver():
  def __init__(self,config,paras,mode):
    self.config=config
    self.paras=paras
    self.mode=mode
    for k, v in default_hparas.items():
      setattr(self,k,v)
    self.device = torch.device(
            'cuda') if self.paras.gpu and torch.cuda.is_available() else torch.device('cpu')
    self.amp=paras.amp

    self.exp_name=paras.name
    if self.exp_name is None:
      self.ext_name=paras.config.split('/')[-1].replace('.yaml','')
      if mode=='train':
        self.exp_name+='_sd{}'.format(paras.seed)

    self.emb_decoder=None

    if mode=='train':
      os.makedirs(paras.ckpdir,exist_ok=True)
      self.ckpdir=os.path.join(paras.ckpdir,self.exp_name)
      os.makedirs(self.ckpdir,exist_ok=True)

      self.logdir=os.path.join(paras.logdir,self.exp_name)
      os.log=SummaryWriter(self.logdir,flush_secs=self.TB_FLUSH_FREQ)
      self.timer=Timer()

      self.step=0
      self.valid_step=config['hparas']['valid_step']
      self.max_step=config['paras']['max_step']
      self.verbose('Exp. name : {}'.format(self.exp_name))
      self.verbose('Loading data... large corpus may took a while.')

    elif mode=='test':
      os.makedirs(paras.outdir,exist_ok=True)
      self.ckpdir=os.path.join(paras.outdir,self.exp_name)

      self.src_config=yaml.load(open(config['src']['config','r']),Loader=yaml.FullLoader)
      self.paras.load=config['src']['ckpt']

      self.verbose('Evaluating result of tr. config @ {}'.format(
                  config['src']['config']))

  def backward(self,loss):
    self.timer.set()
    loss.backward()
    grad_norm=torch.nn.utils.clip_grad_norm(self.model.parameters(),self.GRAD_CLIP)
    if math.isnan(grad_norm):
      self.verbose('Error : grad norm is NaN @ step '+str(self.step))
    else:
      self.optimizer.step()

    self.timer.cnt('bw')
    return grad_norm

  def load_ckpt(self):
    ckpt=torch.load(self.paras.load,map_location=self.device if self.mode=='train' else 'cpu')
    self.model.load_state_dict(ckpt['model'])
    if self.emb_decoder is not None:
      self.emb_decoder.load_state_dict(ckpt['emb_decoder'])
    metric="None"
    score=0.0
    for k, v in ckpt.items():
      if type(v) is float:
        metric,score=k,v
    if self.mode=='train':
      self.step=ckpt['global_step']
      self.optimizer.load_opt_state_dict(ckpt['optimizer'])
      self.verbose('Load ckpt from {}, restarting at step {} (recorded {} = {:.2f} %)'.format(
                              self.paras.load, self.step, metric, score))

    else:
      self.model.eval()
      if self.emb_decoder is not None:
        self.emb_decoder.eval()
      self.verbose('Evaluation target = {} (recorded {} = {:.2f} %)'.format(self.paras.load, metric, score))

  def verbose(self, msg):
    ''' Verbose function for print information to stdout'''
    if self.paras.verbose:
      if type(msg) == list:
        for m in msg:
          print('[INFO]', m.ljust(100))
      else:
        print('[INFO]', msg.ljust(100))

  def progress(self,msg):
    if self.paras.verbose:
      sys.stdout.write("\033[K")  # Clear line
      print('[{}] {}'.format(human_format(self.step), msg), end='\r')

  def write_log(self, log_name, log_dict):
    '''
    Write log to TensorBoard
        log_name  - <str> Name of tensorboard variable
        log_value - <dict>/<array> Value of variable (e.g. dict of losses), passed if value = None
    '''
    if type(log_dict) is dict:
      log_dict = {key: val for key, val in log_dict.items() if (
        val is not None and not math.isnan(val))}
    if log_dict is None:
      pass
    elif len(log_dict) > 0:
      if 'align' in log_name or 'spec' in log_name:
        img, form = log_dict
        self.log.add_image(
          log_name, img, global_step=self.step, dataformats=form)
      elif 'text' in log_name or 'hyp' in log_name:
        self.log.add_text(log_name, log_dict, self.step)
      else:
        self.log.add_scalars(log_name, log_dict, self.step)

  def save_checkpoint(self,f_name,metric,score,show_msg=True):
    ckpt_path=os.path.join(self,ckdir,f_name)
    full_dict={
        "model":self.model.state_dict(),
        "optimizer":self.optimizer.get_opt_state_dict(),
        "global_step":self.step,
        metric:score
    }
    if self.emb_decoder is not None:
      full_dict['emb_decoder']=self.emb_decoder.state_dict()

    torch.save(full_dict,ckpt_path)
    if show_msg:
      self.verbose("Saved checkpoint (step = {}, {} = {:.2f}) and status @ {}".
                         format(human_format(self.step), metric, score, ckpt_path))

  def enable_apex(self):
    if self.amp:
      from apex import amp
      self.amp_lib=amp
      self.verbose(
                "AMP enabled (check https://github.com/NVIDIA/apex for more details).")
      self.model, self.optimizer.opt = self.amp_lib.initialize(
                self.model, self.optimizer.opt, opt_level='O1')

  def load_data(self):
    raise NonImplementedError

  def set_model(self):
    raise NonImplementedError

  def exec(self):
    raise NonImplementedError

# 학습(ASR)

In [ ]:
class Solver(BaseSolver):
    ''' Solver for training'''

    def __init__(self, config, paras, mode):
        super().__init__(config, paras, mode)
        # Logger settings
        self.best_wer = {'att': 3.0, 'ctc': 3.0}
        # Curriculum learning affects data loader
        self.curriculum = self.config['hparas']['curriculum']

    def fetch_data(self, data):
        ''' Move data to device and compute text seq. length'''
        _, feat, feat_len, txt = data
        feat = feat.to(self.device)
        feat_len = feat_len.to(self.device)
        txt = txt.to(self.device)
        txt_len = torch.sum(txt != 0, dim=-1)

        return feat, feat_len, txt, txt_len

    def load_data(self):
        ''' Load data for training/validation, store tokenizer and input/output shape'''
        self.tr_set, self.dv_set, self.feat_dim, self.vocab_size, self.tokenizer, msg = \
            load_dataset(self.paras.njobs, self.paras.gpu, self.paras.pin_memory,
                         self.curriculum > 0, **self.config['data'])
        self.verbose(msg)

    def set_model(self):
        ''' Setup ASR model and optimizer '''
        # Model
        init_adadelta = self.config['hparas']['optimizer'] == 'Adadelta'
        self.model = ASR(self.feat_dim, self.vocab_size, init_adadelta, **
                         self.config['model']).to(self.device)
        self.verbose(self.model.create_msg())
        model_paras = [{'params': self.model.parameters()}]

        # Losses
        self.seq_loss = torch.nn.CrossEntropyLoss(ignore_index=0)
        # Note: zero_infinity=False is unstable?
        self.ctc_loss = torch.nn.CTCLoss(blank=0, zero_infinity=False)

        # Plug-ins
        self.emb_fuse = False
        self.emb_reg = ('emb' in self.config) and (
            self.config['emb']['enable'])
        if self.emb_reg:
            from src.plugin import EmbeddingRegularizer
            self.emb_decoder = EmbeddingRegularizer(
                self.tokenizer, self.model.dec_dim, **self.config['emb']).to(self.device)
            model_paras.append({'params': self.emb_decoder.parameters()})
            self.emb_fuse = self.emb_decoder.apply_fuse
            if self.emb_fuse:
                self.seq_loss = torch.nn.NLLLoss(ignore_index=0)
            self.verbose(self.emb_decoder.create_msg())

        # Optimizer
        self.optimizer = Optimizer(model_paras, **self.config['hparas'])
        self.verbose(self.optimizer.create_msg())

        # Enable AMP if needed
        self.enable_apex()

        # Automatically load pre-trained model if self.paras.load is given
        self.load_ckpt()

        # ToDo: other training methods

    def exec(self):
        ''' Training End-to-end ASR system '''
        self.verbose('Total training steps {}.'.format(
            human_format(self.max_step)))
        ctc_loss, att_loss, emb_loss = None, None, None
        n_epochs = 0
        self.timer.set()

        while self.step < self.max_step:
            # Renew dataloader to enable random sampling
            if self.curriculum > 0 and n_epochs == self.curriculum:
                self.verbose(
                    'Curriculum learning ends after {} epochs, starting random sampling.'.format(n_epochs))
                self.tr_set, _, _, _, _, _ = \
                    load_dataset(self.paras.njobs, self.paras.gpu, self.paras.pin_memory,
                                 False, **self.config['data'])
            for data in self.tr_set:
                # Pre-step : update tf_rate/lr_rate and do zero_grad
                tf_rate = self.optimizer.pre_step(self.step)
                total_loss = 0

                # Fetch data
                feat, feat_len, txt, txt_len = self.fetch_data(data)
                self.timer.cnt('rd')

                # Forward model
                # Note: txt should NOT start w/ <sos>
                ctc_output, encode_len, att_output, att_align, dec_state = \
                    self.model(feat, feat_len, max(txt_len), tf_rate=tf_rate,
                               teacher=txt, get_dec_state=self.emb_reg)

                # Plugins
                if self.emb_reg:
                    emb_loss, fuse_output = self.emb_decoder(
                        dec_state, att_output, label=txt)
                    total_loss += self.emb_decoder.weight*emb_loss

                # Compute all objectives
                if ctc_output is not None:
                    if self.paras.cudnn_ctc:
                        ctc_loss = self.ctc_loss(ctc_output.transpose(0, 1),
                                                 txt.to_sparse().values().to(device='cpu', dtype=torch.int32),
                                                 [ctc_output.shape[1]] *
                                                 len(ctc_output),
                                                 txt_len.cpu().tolist())
                    else:
                        ctc_loss = self.ctc_loss(ctc_output.transpose(
                            0, 1), txt, encode_len, txt_len)
                    total_loss += ctc_loss*self.model.ctc_weight

                if att_output is not None:
                    b, t, _ = att_output.shape
                    att_output = fuse_output if self.emb_fuse else att_output
                    att_loss = self.seq_loss(
                        att_output.view(b*t, -1), txt.view(-1))
                    total_loss += att_loss*(1-self.model.ctc_weight)

                self.timer.cnt('fw')

                # Backprop
                grad_norm = self.backward(total_loss)
                self.step += 1

                # Logger
                if (self.step == 1) or (self.step % self.PROGRESS_STEP == 0):
                    self.progress('Tr stat | Loss - {:.2f} | Grad. Norm - {:.2f} | {}'
                                  .format(total_loss.cpu().item(), grad_norm, self.timer.show()))
                    self.write_log(
                        'loss', {'tr_ctc': ctc_loss, 'tr_att': att_loss})
                    self.write_log('emb_loss', {'tr': emb_loss})
                    self.write_log('wer', {'tr_att': cal_er(self.tokenizer, att_output, txt),
                                           'tr_ctc': cal_er(self.tokenizer, ctc_output, txt, ctc=True)})
                    if self.emb_fuse:
                        if self.emb_decoder.fuse_learnable:
                            self.write_log('fuse_lambda', {
                                           'emb': self.emb_decoder.get_weight()})
                        self.write_log(
                            'fuse_temp', {'temp': self.emb_decoder.get_temp()})

                # Validation
                if (self.step == 1) or (self.step % self.valid_step == 0):
                    self.validate()

                # End of step
                # https://github.com/pytorch/pytorch/issues/13246#issuecomment-529185354
                torch.cuda.empty_cache()
                self.timer.set()
                if self.step > self.max_step:
                    break
            n_epochs += 1
        self.log.close()

    def validate(self):
        # Eval mode
        self.model.eval()
        if self.emb_decoder is not None:
            self.emb_decoder.eval()
        dev_wer = {'att': [], 'ctc': []}

        for i, data in enumerate(self.dv_set):
            self.progress('Valid step - {}/{}'.format(i+1, len(self.dv_set)))
            # Fetch data
            feat, feat_len, txt, txt_len = self.fetch_data(data)

            # Forward model
            with torch.no_grad():
                ctc_output, encode_len, att_output, att_align, dec_state = \
                    self.model(feat, feat_len, int(max(txt_len)*self.DEV_STEP_RATIO),
                               emb_decoder=self.emb_decoder)

            dev_wer['att'].append(cal_er(self.tokenizer, att_output, txt))
            dev_wer['ctc'].append(cal_er(self.tokenizer, ctc_output, txt, ctc=True))

            # Show some example on tensorboard
            if i == len(self.dv_set)//2:
                for i in range(min(len(txt), self.DEV_N_EXAMPLE)):
                    if self.step == 1:
                        self.write_log('true_text{}'.format(
                            i), self.tokenizer.decode(txt[i].tolist()))
                    if att_output is not None:
                        self.write_log('att_align{}'.format(i), feat_to_fig(
                            att_align[i, 0, :, :].cpu().detach()))
                        self.write_log('att_text{}'.format(i), self.tokenizer.decode(
                            att_output[i].argmax(dim=-1).tolist()))
                    if ctc_output is not None:
                        self.write_log('ctc_text{}'.format(i), self.tokenizer.decode(ctc_output[i].argmax(dim=-1).tolist(),
                                                                                     ignore_repeat=True))

        # Ckpt if performance improves
        for task in ['att', 'ctc']:
            dev_wer[task] = sum(dev_wer[task])/len(dev_wer[task])
            if dev_wer[task] < self.best_wer[task]:
                self.best_wer[task] = dev_wer[task]
                self.save_checkpoint('best_{}.pth'.format(task), 'wer', dev_wer[task])
            self.write_log('wer', {'dv_'+task: dev_wer[task]})
        self.save_checkpoint('latest.pth', 'wer', dev_wer['att'], show_msg=False)

        # Resume training
        self.model.train()
        if self.emb_decoder is not None:
            self.emb_decoder.train()

#학습(LM)

In [ ]:

class Solver(BaseSolver):
    ''' Solver for training language models'''

    def __init__(self, config, paras, mode):
        super().__init__(config, paras, mode)
        # Logger settings
        self.best_loss = 10

    def fetch_data(self, data):
        ''' Move data to device, insert <sos> and compute text seq. length'''
        txt = torch.cat(
            (torch.zeros((data.shape[0], 1), dtype=torch.long), data), dim=1).to(self.device)
        txt_len = torch.sum(data != 0, dim=-1)
        return txt, txt_len

    def load_data(self):
        ''' Load data for training/validation, store tokenizer and input/output shape'''
        self.tr_set, self.dv_set, self.vocab_size, self.tokenizer, msg = \
            load_textset(self.paras.njobs, self.paras.gpu,
                         self.paras.pin_memory, **self.config['data'])
        self.verbose(msg)

    def set_model(self):
        ''' Setup ASR model and optimizer '''

        # Model
        self.model = RNNLM(self.vocab_size, **
                           self.config['model']).to(self.device)
        self.verbose(self.model.create_msg())
        # Losses
        self.seq_loss = torch.nn.CrossEntropyLoss(ignore_index=0)
        # Optimizer
        self.optimizer = Optimizer(
            self.model.parameters(), **self.config['hparas'])
        # Enable AMP if needed
        self.enable_apex()
        # load pre-trained model
        if self.paras.load:
            self.load_ckpt()
            ckpt = torch.load(self.paras.load, map_location=self.device)
            self.model.load_state_dict(ckpt['model'])
            self.optimizer.load_opt_state_dict(ckpt['optimizer'])
            self.step = ckpt['global_step']
            self.verbose('Load ckpt from {}, restarting at step {}'.format(
                self.paras.load, self.step))

    def exec(self):
        ''' Training End-to-end ASR system '''
        self.verbose('Total training steps {}.'.format(
            human_format(self.max_step)))
        self.timer.set()

        while self.step < self.max_step:
            for data in self.tr_set:
                # Pre-step : update tf_rate/lr_rate and do zero_grad
                self.optimizer.pre_step(self.step)

                # Fetch data
                txt, txt_len = self.fetch_data(data)
                self.timer.cnt('rd')

                # Forward model
                pred, _ = self.model(txt[:, :-1], txt_len)

                # Compute all objectives
                lm_loss = self.seq_loss(
                    pred.view(-1, self.vocab_size), txt[:, 1:].reshape(-1))
                self.timer.cnt('fw')

                # Backprop
                grad_norm = self.backward(lm_loss)
                self.step += 1

                # Logger
                if self.step % self.PROGRESS_STEP == 0:
                    self.progress('Tr stat | Loss - {:.2f} | Grad. Norm - {:.2f} | {}'
                                  .format(lm_loss.cpu().item(), grad_norm, self.timer.show()))
                    self.write_log('entropy', {'tr': lm_loss})
                    self.write_log(
                        'perplexity', {'tr': torch.exp(lm_loss).cpu().item()})

                # Validation
                if (self.step == 1) or (self.step % self.valid_step == 0):
                    self.validate()

                # End of step
                self.timer.set()
                if self.step > self.max_step:
                    break
        self.log.close()

    def validate(self):
        # Eval mode
        self.model.eval()
        dev_loss = []

        for i, data in enumerate(self.dv_set):
            self.progress('Valid step - {}/{}'.format(i+1, len(self.dv_set)))
            # Fetch data
            txt, txt_len = self.fetch_data(data)

            # Forward model
            with torch.no_grad():
                pred, _ = self.model(txt[:, :-1], txt_len)
            lm_loss = self.seq_loss(
                pred.view(-1, self.vocab_size), txt[:, 1:].reshape(-1))
            dev_loss.append(lm_loss)

        # Ckpt if performance improves
        dev_loss = sum(dev_loss)/len(dev_loss)
        dev_ppx = torch.exp(dev_loss).cpu().item()
        if dev_loss < self.best_loss:
            self.best_loss = dev_loss
            self.save_checkpoint('best_ppx.pth', 'perplexity', dev_ppx)
        self.write_log('entropy', {'dv': dev_loss})
        self.write_log('perplexity', {'dv': dev_ppx})

        # Show some example of last batch on tensorboard
        for i in range(min(len(txt), self.DEV_N_EXAMPLE)):
            if self.step == 1:
                self.write_log('true_text{}'.format(
                    i), self.tokenizer.decode(txt[i].tolist()))
            self.write_log('pred_text{}'.format(i), self.tokenizer.decode(
                pred[i].argmax(dim=-1).tolist()))

        # Resume training
        self.model.train()

# 테스트(ASR)

In [ ]:

class Solver(BaseSolver):
    ''' Solver for training'''

    def __init__(self, config, paras, mode):
        super().__init__(config, paras, mode)

        # ToDo : support tr/eval on different corpus
        assert self.config['data']['corpus']['name'] == self.src_config['data']['corpus']['name']
        self.config['data']['corpus']['path'] = self.src_config['data']['corpus']['path']
        self.config['data']['corpus']['bucketing'] = False

        # The follow attribute should be identical to training config
        self.config['data']['audio'] = self.src_config['data']['audio']
        self.config['data']['text'] = self.src_config['data']['text']
        self.config['hparas'] = self.src_config['hparas']
        self.config['model'] = self.src_config['model']

        # Output file
        self.output_file = str(self.ckpdir)+'_{}_{}.csv'

        # Override batch size for beam decoding
        self.greedy = self.config['decode']['beam_size'] == 1
        if not self.greedy:
            self.config['data']['corpus']['batch_size'] = 1

        self.step = 0

    def fetch_data(self, data):
        ''' Move data to device and compute text seq. length,
            For Greedy decoding only ( beam_decode & ctc_beam_decode otherwise)'''
        _, feat, feat_len, txt = data
        feat = feat.to(self.device)
        feat_len = feat_len.to(self.device)
        txt = txt.to(self.device)
        txt_len = torch.sum(txt!=0,dim=-1)

        return feat, feat_len, txt, txt_len

    def load_data(self):
        ''' Load data for training/validation, store tokenizer and input/output shape'''
        self.dv_set, self.tt_set, self.feat_dim, self.vocab_size, self.tokenizer, msg = \
            load_dataset(self.paras.njobs, self.paras.gpu,
                         self.paras.pin_memory, False, **self.config['data'])
        self.verbose(msg)

    def set_model(self):
        ''' Setup ASR model '''
        # Model
        init_adadelta = self.config['hparas']['optimizer'] == 'Adadelta'
        self.model = ASR(self.feat_dim, self.vocab_size, init_adadelta, **
                         self.config['model']).to(self.device)

        # Plug-ins
        if ('emb' in self.config) and (self.config['emb']['enable']) \
                and (self.config['emb']['fuse'] > 0):
            from src.plugin import EmbeddingRegularizer
            self.emb_decoder = EmbeddingRegularizer(
                self.tokenizer, self.model.dec_dim, **self.config['emb'])

        # Load target model in eval mode
        self.load_ckpt()

        self.ctc_only = False
        if self.greedy:
            # Greedy decoding: attention-based if the ASR has a decoder, else use CTC
            self.decoder = copy.deepcopy(self.model).to(self.device)
        else:
            if (not self.model.enable_att) or self.config['decode'].get('ctc_weight', 0.0) == 1.0:
                # Pure CTC Beam Decoder
                assert self.config['decode']['beam_size'] <= self.config['decode']['vocab_candidate']
                self.decoder = CTCBeamDecoder(self.model.to(self.device),
                    [1] + [r for r in range(3, self.vocab_size)],
                    self.config['decode']['beam_size'],
                    self.config['decode']['vocab_candidate'],
                    lm_path=self.config['decode']['lm_path'],
                    lm_config=self.config['decode']['lm_config'],
                    lm_weight=self.config['decode']['lm_weight'],
                    device=self.device)
                self.ctc_only = True
            else:
                # Joint CTC-Attention Beam Decoder
                self.decoder = BeamDecoder(
                    self.model.cpu(), self.emb_decoder, **self.config['decode'])

        self.verbose(self.decoder.create_msg())
        del self.model
        del self.emb_decoder

    def greedy_decode(self, dv_set):
        ''' Greedy Decoding '''
        results = []
        for i,data in enumerate(dv_set):
            self.progress('Valid step - {}/{}'.format(i+1,len(dv_set)))
            # Fetch data
            feat, feat_len, txt, txt_len = self.fetch_data(data)
            # Forward model
            with torch.no_grad():
                ctc_output, encode_len, att_output, att_align, dec_state = \
                    self.decoder( feat, feat_len, int(float(feat_len.max()) * self.config['decode']['max_len_ratio']),
                                    emb_decoder=self.emb_decoder)
            for j in range(len(txt)):
                idx = j + self.config['data']['corpus']['batch_size'] * i
                if att_output is not None:
                    hyp_seqs = att_output[j].argmax(dim=-1).tolist()
                else:
                    hyp_seqs = ctc_output[j].argmax(dim=-1).tolist()
                true_txt = txt[j]
                results.append((str(idx), [hyp_seqs], true_txt))
        return results

    def exec(self):
        ''' Testing End-to-end ASR system '''
        for s, ds in zip(['dev','test'],[self.dv_set,self.tt_set]):
            # Setup output
            self.cur_output_path = self.output_file.format(s,'output')
            with open(self.cur_output_path,'w',encoding='UTF-8') as f:
                f.write('idx\thyp\ttruth\n')

            if self.greedy:
                # Greedy decode
                self.verbose(
                    'Performing batch-wise greedy decoding on {} set, num of batch = {}.'.format(s, len(ds)))
                results = self.greedy_decode(ds)
                self.verbose('Results will be stored at {}'.format(
                    self.cur_output_path))
                self.write_hyp(results, self.cur_output_path, '-')
            elif self.ctc_only:
                # CTC beam decode
                # Additional output to store all beams
                self.cur_beam_path = self.output_file.format(s,
                    'beam-{}-{}'.format(self.config['decode']['beam_size'], self.config['decode']['lm_weight']))
                with open(self.cur_beam_path,'w') as f:
                    f.write('idx\tbeam\thyp\ttruth\n')
                self.verbose('Performing instance-wise CTC beam decoding on {} set, num of batch = {}.'.format(s,len(ds)))
                # Minimal function to pickle
                ctc_beam_decode_func = partial(ctc_beam_decode, model=copy.deepcopy(self.decoder), device=self.device)
                # Parallel beam decode
                results = Parallel(n_jobs=self.paras.njobs)(delayed(ctc_beam_decode_func)(data) for data in tqdm(ds))
                self.verbose('Results/Beams will be stored at {} / {}'.format(self.cur_output_path,self.cur_beam_path))
                self.write_hyp(results, self.cur_output_path, self.cur_beam_path)
            else:
                # Joint CTC-Attention beam decode
                # Additional output to store all beams
                self.cur_beam_path = self.output_file.format(s,
                    'beam-{}-{}'.format(self.config['decode']['beam_size'], self.config['decode']['lm_weight']))
                with open(self.cur_beam_path, 'w') as f:
                    f.write('idx\tbeam\thyp\ttruth\n')
                self.verbose(
                    'Performing instance-wise beam decoding on {} set. (NOTE: use --njobs to speedup)'.format(s))
                # Minimal function to pickle
                beam_decode_func = partial(beam_decode, model=copy.deepcopy(
                    self.decoder), device=self.device)
                # Parallel beam decode
                results = Parallel(n_jobs=self.paras.njobs)(
                    delayed(beam_decode_func)(data) for data in tqdm(ds))
                self.verbose(
                    'Results/Beams will be stored at {} / {}.'.format(self.cur_output_path, self.cur_beam_path))
                self.write_hyp(results, self.cur_output_path,
                               self.cur_beam_path)
        self.verbose('All done !')

    def write_hyp(self, results, best_path, beam_path):
        '''Record decoding results'''
        if self.greedy:
            # Ignores repeated symbols if is decoded with CTC
            ignore_repeat = not self.decoder.enable_att
        else:
            ignore_repeat = False
        for name, hyp_seqs, truth in tqdm(results):
            if self.ctc_only and not self.greedy:
                new_hyp_seqs = [self.tokenizer.decode(hyp, ignore_repeat=False) for hyp in hyp_seqs[:-1]]
                hyp_seqs = new_hyp_seqs + [self.tokenizer.decode(hyp_seqs[-1], ignore_repeat=True)]
            else:
                hyp_seqs = [self.tokenizer.decode(hyp) for hyp in hyp_seqs]

            truth = self.tokenizer.decode(truth)
            with open(best_path, 'a') as f:
                if len(hyp_seqs[0]) == 0:
                    # Set the sequence to a whitespace if it was empty
                    hyp_seqs[0] = ' '
                f.write('\t'.join([name, hyp_seqs[0], truth])+'\n')
            if not self.greedy:
                with open(beam_path, 'a', encoding='UTF-8') as f:
                    for b, hyp in enumerate(hyp_seqs):
                        f.write('\t'.join([name, str(b), hyp, truth])+'\n')

def beam_decode(data, model, device):
    # Fetch data : move data/model to device
    name, feat, feat_len, txt = data
    feat = feat.to(device)
    feat_len = feat_len.to(device)
    txt = txt.to(device)
    txt_len = torch.sum(txt != 0, dim=-1)
    model = model.to(device)
    # Decode
    with torch.no_grad():
        hyps = model(feat, feat_len)

    hyp_seqs = [hyp.outIndex for hyp in hyps]
    del hyps
    return (name[0], hyp_seqs, txt[0].cpu().tolist())  # Note: bs == 1

def ctc_beam_decode(data, model, device):
    # Fetch data : move data/model to device
    name, feat, feat_len, txt = data
    feat = feat.to(device)
    feat_len = feat_len.to(device)
    # Decode
    with torch.no_grad():
        hyp = model(feat, feat_len)

    return (name[0], hyp, txt[0]) # Note: bs == 1

# 하이퍼파라미터 및 실행

In [37]:
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

In [38]:
parser = argparse.ArgumentParser(description='Training E2E asr.')

parser.add_argument('--config', type=str, help='Path to experiment config.')
parser.add_argument('--name', default=None, type=str, help='Name for logging.')
parser.add_argument('--logdir', default='log/', type=str,
                    help='Logging path.', required=False)
parser.add_argument('--ckpdir', default='ckpt/', type=str,
                    help='Checkpoint path.', required=False)
parser.add_argument('--outdir', default='result/', type=str,
                    help='Decode output path.', required=False)
parser.add_argument('--load', default=None, type=str,
                    help='Load pre-trained model (for training only)', required=False)
parser.add_argument('--seed', default=0, type=int,
                    help='Random seed for reproducable results.', required=False)
parser.add_argument('--cudnn-ctc', action='store_true',
                    help='Switches CTC backend from torch to cudnn')
parser.add_argument('--njobs', default=6, type=int,
                    help='Number of threads for dataloader/decoding.', required=False)
parser.add_argument('--cpu', action='store_true', help='Disable GPU training.')
parser.add_argument('--no-pin', action='store_true',
                    help='Disable pin-memory for dataloader')
parser.add_argument('--test', action='store_true', help='Test the model.')
parser.add_argument('--no-msg', action='store_true', help='Hide all messages.')
parser.add_argument('--lm', action='store_true',
                    help='Option for training RNNLM.')
# Following features in development.
parser.add_argument('--amp', action='store_true', help='Option to enable AMP.')
parser.add_argument('--reserve-gpu', default=0, type=float,
                    help='Option to reserve GPU ram for training.')
parser.add_argument('--jit', action='store_true',
                    help='Option for enabling jit in pytorch. (feature in development)')

_StoreTrueAction(option_strings=['--jit'], dest='jit', nargs=0, const=True, default=False, type=None, choices=None, required=False, help='Option for enabling jit in pytorch. (feature in development)', metavar=None)

In [39]:
paras, unknown = parser.parse_known_args()
setattr(paras,'gpu',not paras.cpu)
setattr(paras,'pin_memory',not paras.no_pin)
setattr(paras,'verbose',not paras.no_msg)
config=yaml.load(open(paras.config,'r'),Loader=yaml.FullLoader)

np.random.seed(paras.seed)
torch.manual_seed(paras.seed)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(paras.seed)

if paras.gpu and paras.reserve_gpu>0:
  buff=torch.randn(int(paras.reserve_gpu*1e9//4)).cuda()

if paras.lm:
  mode='train'

else:
  if paras.test:
    assert paras.load is None, 'Load option is mutually exclusive to --test'

    mode = 'test'




TypeError: expected str, bytes or os.PathLike object, not NoneType

# 최종 실행

In [ ]:
solver=Solver(config,paras,mode)
solver.load_data()
solver.set_model()
solver.exec()